In [ ]:
import h5py
import glob
import os
from tqdm import tqdm
import math 
import glob
    
import tensorflow as tf
from tensorflow.keras.layers import BatchNormalization, MaxPool2D, Conv2D, Flatten, LSTM, Dense, TimeDistributed, Activation
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import LearningRateScheduler

import numpy as np
from numpy import load

import random

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import clear_output
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.cluster import KMeans 
from scipy.stats import multivariate_normal
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from IPython.display import clear_output

In [ ]:
if 'model' in locals(): del model
if 'full_model' in locals(): del full_model

def set_seeds(seed=69):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    # If using GPU, this ensures deterministic operations (can be slower)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'

set_seeds(69)

## Our Data

Let's start by describing our dataset, SoliDataset is composed of 5,500 files of h5 type, which is a type of Hierarchical Data Format. The files were obtained by Soli sensor, which is a solid-state milimeter-wave radar for mobile gesture recognitions. The radar has 4 antennes, so each file is composed by 4 channels, Since we are working with gestures, each file contains a sequence of frames that represents every step of the motion.

When extracting the data, each file contains 5 keys: ch1, ch2, ch3, ch4, label, which refers to the channel 1 to channel 4 and the gesture corresponding to the sequence, respectively. Each channel contains different amount of frames, and each of this frames has 1,024 elements corresponding to the pixels of the image, lately these will be reshape to a 32x32 structure.

From the 5,500 files, half of them correspond to Images recorded at 40Hz that captured each gesture 25 times from all 11 subjects over 10 sessions resulting in 11 (gesture) x 25 (times) x 10 (sessions) = 2,750, among all users

For cross-user evaluation we will consider: session 2 (25), 3 (25), 5 (25), 6 (25), 8 (25), 9 (25), 10 (25),11 (25), 12 (25), 13 (25)

The rest of the files were recorded with the purpose of evaluating cross-session performance and to explore personalized gesture recognition. So another 11 (gestures) x 50 (times) x 5 (sessions) = 2,750 sequences from a single user were obtained.

For cross-session evaluation session 0 (50), 1 (50), 4 (50), 7 (50), 13 (25), 14 (25)

During the following section we will modify the files' structure to create tensor of the following dimensions: (F, C, H, W), representing number of frames, channels, height and width of the image, respectively. Posterior to this, we will truncate the sequences that contain more than 40 frames and we will pad the ones with less than 40. This is made with the purpose of creating batches of data, which requires consistency of the dimensions. For this reason, in the end we will work with data with the following dimensions (40, 32, 32, 4).

In [ ]:
#Function to visualize an RD map from a single file

def visualize_gesture_progression(file_path, channel='ch0', steps=5):
    with h5py.File(file_path, 'r') as f:
        data = np.array(f[channel]) # Shape (t, 1024)
        t = data.shape[0]
        indices = np.linspace(0, t-1, steps, dtype=int)
        
    fig, axes = plt.subplots(1, steps, figsize=(20, 4))
    fig.suptitle(f"Progression of {channel} over {t} frames", fontsize=16)
    
    for i, idx in enumerate(indices):
        rd_map = data[idx].reshape(32, 32)
        axes[i].imshow(rd_map, cmap='viridis', origin='lower')
        axes[i].set_title(f"Frame {idx}")
        axes[i].axis('off')
        
    plt.show()

In [ ]:
file_path = '/kaggle/input/solidata/dsp/0_10_10.h5'

visualize_gesture_progression(file_path, channel = 'ch2', steps = 5)

# Preprocessing Helper Functions

In this section, all the code to preprocess the data. We first define a GMM class for image segmentation in n classes that can be fitted to an image, and used to predict labels on that or other images (credits + our modification). Since we set n = 2, this becomes segmentation between hand/ background, and we use this to remove the background of all frames of all sequences in the dataset. The preprocessing functions are as follows:

1. Preprocessing a single file: This function first fetches a h5py file, unpacks it, applies the gmm.fit() method to the first frame of a sequence (instead of every frame, as this would be costly computationally), and creates a mask of background and hand classifications. This mask is then applied by multiplying to the original image, where all the background pixels are multiplied to 0.

2. Creating a new dataset of preprocessed data: Since preprocessing a single file involves fitting a GMM, this is costly. To run this every epoch in training as the dataset object gets called would be extremely costly, so instead we preprocess the data once and save it to a directory in a compressed numpy format that is then used to continue the data pipeline.

3. Helper functions for more dataset preprocessing: The largest chunk of preprocessing corresponds to the GMM background removal, but data is still not input-ready. We created some helper functions to adjust the sequences to a constant length by padding or truncating with the option of taking the middle t frames, the first t frames, or a random crop of t frames. We also have a helper function to clip the RDIs, as in Tsang21 they clip RDIs to the bottom 12 pixels.

4. Creating tensorflow dataset iterators: We have a separate functions to create the dataset iterators for the CNNRNN model and the LSM model, as they need different preprocessing. For the CNNRNN dataset iterator, we have the option to apply either of the helper functions described above, and also we normalise the signals, with the option of applying minmax normalization or log normalization. For the LSM dataset iterator, we also have the option to apply either of the helper functions, but we also convert the radar signal into a spike trains, that is, binary data, and we flatten the input to (t, HxWxC).

In [ ]:
#GMM definition

class GMM(object):
    
    weights = None
    means = None
    covars = None
    k=None
    iterations =100
    convergence_th=1e-3
    ric=None
    
    def __init__(self, n_components=1, tol=1e-3, max_iter=100):
        """
        A Gaussian mixture model trained via the expectation maximization
        algorithm.
        
        Parameters
        ----------
        
        n_components: The number of mixture components.
        tol: The convergence threshold
        """
        self.k=n_components
        self.iterations=max_iter
        self.convergence_th=tol


    def initialize_params(self, X, kmeans=False):
        """
        Initialize the starting GMM parameters.
        
        Parameters
        ----------
        X : A collection of `N` training data points, each with dimension `d`.
        kmeans: A boolean flag for determining if to initialize the params with kmeans or randomly.
        """

        self.means = np.random.rand(self.k,X.shape[1])
        self.covars = np.zeros((self.k, X.shape[1], X.shape[1]))
        for k in range(self.k):
            self.covars[k] = np.eye(X.shape[1])
        self.weights=[1/self.k]*self.k
        self.ric = np.zeros((X.shape[0], self.k))
    
    def formula(self,x,u,sigma,d):
        first_term = 1/(((2*np.pi)**(d/2))*np.sqrt(sigma))
        exp = -0.5*(np.dot(np.dot((x-u).T,np.linalg.inv(sigma)),(x-u)))
        second_term = np.exp(exp)
        return first_term*second_term
    
    def E_step(self, X):
        """
        Find the Expectation of the log-likelihood evaluated using the current estimate for the parameters
        
        Parameters
        ----------
        X : A collection of `N` data points, each with dimension `d`.
        """
        n=X.shape[0]
        d = X.shape[1]

        nan_mask = np.isnan(self.covars)
        inf_mask = np.isinf(self.covars)
        self.covars[nan_mask] = 1e-6
        self.covars[inf_mask] = 1e-6
        
        for c in range(self.k):
            self.ric[:,c]= self.weights[c]*multivariate_normal.pdf(X,mean=self.means[c],cov=self.covars[c],allow_singular=True)
        for i in range(n):
            row_sum = np.sum(self.ric[i])
            if row_sum > 0:
                self.ric[i] = self.ric[i]/row_sum
            else:
                self.ric[i] = np.ones(self.k)/self.k
        return

    def M_step(self, X):
        """
        Updates parameters maximizing the expected log-likelihood found on the E step.
        
        Parameters
        ----------
        X : A collection of `N` data points, each with dimension `d`.
        """
        n=X.shape[0]
        d=X.shape[1]

        for c in range(self.k):
            self.weights[c]=np.sum(self.ric[:,c])/n
            
        for c in range(self.k):
            clusterSum = np.sum(self.ric[:, c])
            self.means[c] = np.sum(X * self.ric[:, c][:, np.newaxis], axis=0) / clusterSum
            diff = X - self.means[c]
            
            reg_term = 1e-6*np.eye(d)
            self.covars[c] = (np.dot(self.ric[:, c] * diff.T, diff)/ clusterSum) + reg_term 
            self.weights[c] = clusterSum / n

        

    def fit(self, X, y=None):
        """
        Fit the parameters of the GMM on some training data.
        
        Parameters
        ----------
        X : A collection of `N` training data points, each with dimension `d`.
        y: not used
        """
        self.initialize_params(X)
        for i in range(self.iterations):
            self.E_step(X)
            self.M_step(X)
        
    def predict(self, X):
        """
        Predict the labels for the data samples in X using trained model.
        
        Parameters
        ----------
        X : A collection of `M` data points, each with dimension `d`.
        
        Returns
        -------
        Predicted labels.
        """
        N = X.shape[0]
        gamma = np.zeros((N, self.k))
        for k in range(self.k):
            pdf = multivariate_normal(mean=self.means[k], cov=self.covars[k])
            gamma[:, k] = self.weights[k] * pdf.pdf(X)
        return np.argmax(gamma, axis=1)

In [ ]:
#Function to preprocess a single h5py file with GMM background removal

def process_soli_file(file_path, target_frames=40):
    with h5py.File(file_path, 'r') as f:
        # 1. Extract channels and stack: (t, 1024, 4)
        # We assume ch0, ch1, ch2, ch3 all have the same 't'
        ch_data = [np.array(f[f'ch{i}']) for i in range(4)]
        data = np.stack(ch_data, axis=-1) # Shape: (t, 1024, 4)
        label = f['label'][0]

    # 2. GMM Background Removal (Frame-by-Frame)
    t, bins, ch = data.shape
    X_all = data.reshape(-1, ch)

    gmm = GMM(n_components = 2, max_iter = 10)
    gmm.fit(X_all)
    labels = gmm.predict(X_all)

    energy = np.linalg.norm(X_all, axis=1)

    mean_energy_cluster_0 = np.mean(energy[labels == 0])
    mean_energy_cluster_1 = np.mean(energy[labels == 1])

    if mean_energy_cluster_0 > mean_energy_cluster_1:
        hand_label = 0
    else:
        hand_label = 1

    mask = (labels == hand_label).reshape(t, 1024, 1).astype(np.float32)

    sequence = (data * mask).reshape(t,32,32,4)

    return sequence.astype(np.float32), int(label)

In [ ]:
# Preprocessing function to save to directory, as preprocessing is slow, to save on model training time

def precompute_dataset(file_paths, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    for path in tqdm(file_paths, desc="Processing GMM"):
        
        processed_data, label = process_soli_file(path, target_frames=40)
        
        # Copy filename with different extension
        base_name = os.path.basename(path).replace('.h5', '.npz')
        out_path = os.path.join(output_dir, base_name)
        
        #Save as compressed numpy file
        np.savez_compressed(out_path, x=processed_data, y=label)

In [ ]:
#Create new folder with pre-processed data, only applying the GMMM background removal
#Only needed to run this once

"""
dataset_path = "/kaggle/input/solidata/dsp"
all_file_paths_h5 = sorted(glob.glob(os.path.join(dataset_path, "**/*.h5"), recursive=True))

preprocessed_dir = "./data_preprocessed"
precompute_dataset(all_file_paths_h5, preprocessed_dir)
"""

In [ ]:
def apply_clipping(sequence, clip_y=True, clip_amount=20):
    if not clip_y:
        return sequence
    # Removes top 20 rows (Range bins) assuming y-axis is Range
    # sequence shape is (t, range, doppler, ch)
    return sequence[:, clip_amount:, :, :]

def apply_padding_truncating(sequence, target_frames, mode='first'):
    """
    mode options: 'first', 'middle', 'random'
    """
    t_current = sequence.shape[0]
    
    # --- Truncation Logic ---
    if t_current > target_frames:
        if mode == 'middle':
            start = (t_current - target_frames) // 2
            return sequence[start : start + target_frames]
        elif mode == 'random':
            start = np.random.randint(0, t_current - target_frames + 1)
            return sequence[start : start + target_frames]
        else: # 'first'
            return sequence[:target_frames]
            
    # --- Padding Logic ---
    else:
        pad_shape = (target_frames - t_current,) + sequence.shape[1:]
        padding = np.zeros(pad_shape, dtype=np.float32)
        # For simplicity, we pad at the end regardless of mode, 
        # but you could center the data for 'middle' mode here too.
        return np.concatenate([sequence, padding], axis=0)

In [ ]:
def get_cnn_rnn_dataset(file_path_list, batch_size = 8, target_frames = 40, norm_type = 'minmax',
                       trunc_mode = 'middle', clip_range = True, clip_amount = 20, shuffle = False):

    file_paths = file_path_list
    
    def generator():
        if shuffle: np.random.shuffle(file_paths)
        for path in file_paths:
            with np.load(path) as f:
                X, y = f['x'], f['y']
                if y == 11: continue

                if clip_range:
                    X = apply_clipping(X, clip_range, clip_amount)

                if norm_type == 'log':
                    X = np.log1p(X)
                elif norm_type == 'minmax':
                    denom = np.max(X) - np.min(X)
                    if denom > 1e-6: X = (X - np.min(X)) / denom

                X = apply_padding_truncating(X, target_frames, mode = trunc_mode)
                yield X, y

    range_dim = 32 - clip_amount if clip_range else 32
    output_signature = (
        tf.TensorSpec(shape = (target_frames, range_dim, 32, 4), dtype = tf.float32),
        tf.TensorSpec(shape = (), dtype = tf.int32)
    )

    return tf.data.Dataset.from_generator(generator, output_signature = output_signature).batch(batch_size).prefetch(tf.data.AUTOTUNE)
                

In [ ]:
def get_lsm_dataset(file_path_list, batch_size = 8, apply_temporal = True, target_frames = 40,
                   trunc_mode = 'middle', threshold = 0, clip_range = True, clip_amount = 20, shuffle = False):

    file_paths = list(file_path_list)
    
    def generator():
        
        if shuffle: np.random.shuffle(file_paths)
            
        for path in file_paths:
            with np.load(path) as f:
                X, y = f['x'], f['y']
                if y == 11: continue

                if clip_range:
                    X = apply_clipping(X, clip_range, clip_amount)

                if apply_temporal: 
                    X = apply_padding_truncating(X, target_frames, mode = trunc_mode)

                spikes = (X > threshold).astype(np.float32)

                t_dim = X.shape[0]
                spikes_flat = spikes.reshape(t_dim, -1)

                yield spikes_flat, y

    t_shape = target_frames if apply_temporal else None
    range_dim = 32 - clip_amount if clip_range else 32
    flat_dim = range_dim * 32 * 4

    output_signature = (
        tf.TensorSpec(shape = (t_shape, flat_dim), dtype = tf.float32),
        tf.TensorSpec(shape = (), dtype = tf.int32)
    )

    ds = tf.data.Dataset.from_generator(generator, output_signature = output_signature)

    if not apply_temporal:
        return ds.padded_batch(batch_size).prefetch(tf.data.AUTOTUNE)
    else:
        return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


In [ ]:
# # Old Function to create the tensorflow Dataset object

# def get_preprocessed_dataset(folder_path, batch_size=8, shuffle=False):

#     file_paths = glob.glob(os.path.join(folder_path, "*.npz"))

#     if not file_paths:
#         raise ValueError(f"No .npz files found in {folder_path}")
    
#     def generator():

#         if shuffle:
#             np.random.shuffle(file_paths)
        
#         for path in file_paths:
#             # Using our new fast GMM logic
#             try:
#                 with np.load(path) as f:
#                     X = f['x'].astype(np.float32)
#                     y = f['y'].astype(np.int32)
#                 yield X, y
#             except Exception as e:
#                 print(f"Skipping {path} due to error: {e}")
#                 continue

#     output_signature = (
#         tf.TensorSpec(shape=(40, 32, 32, 4), dtype=tf.float32),
#         tf.TensorSpec(shape=(), dtype=tf.int32)
#     )

#     dataset = tf.data.Dataset.from_generator(
#         generator,
#         output_signature=output_signature
#     )

#     dataset = dataset.filter(lambda x, y: y != 11)
   
#     if shuffle:
#         dataset = dataset.shuffle(buffer_size=len(file_paths))
            
#     dataset = dataset.batch(batch_size)
#     dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
#     return dataset

## Data Pipeline

In this section, we apply the pipeline functions that we defined previously, and we include health checks along the way to make sure the data is of the shapes and values we need it to be to be input-ready. The pipeline flows as follows:

1. Create a list of all the file paths in the processed dataset. From this main list we can then create train / val/ test splits and apply cross validation in training. Since file names contain information about individual subjects, sessions and gestures, we can also filter and cross validate using these parameters with the list of file paths.
2. Health check: Apply manually GMM background removal to an image fetched from the original dataset, compare before and after to check that GMM works correctly.
3. Health check: Fetch an image from the original dataset, and then the same image from the preprocessed dataset to compare and ensure the preprocessed folder is correctly preprocessed.
4. Create the tensorflow dataset iterators for both models. It is important to note that these take as input the list of files, so it is easy to apply cross validation as we can use loops of creating these iterators with different train/ val splits of file lists.
5. Health check: For the CNNRNN dataset iterator, check the statistics of the non-zero (non-background) values and see that they are in a healthy range to be input into the model (With Relu Activation, we want values between 0 and 1, but not too small that the gradient might vanish). For the LSM iterator, check that it only contains binary values (0s and 1s) and is of the correct shape (batch, t, HxWxC)
6. Health check: For the CNNRNN iterator, fetch sequences for each of the train/ val/ test iterators. Check if they are correct shape (batch, t, 32, 32, 4) and that visually they make sense as RDIs. This kind of check doesn't make sense for the LSM iterator as this dataset contains only flattened temporal spike trains.

In [ ]:
#First, create the list of all file paths. This list will be used to create train/val/test splits

dataset_path = '/kaggle/input/train-preprocessed/data_preprocessed'

all_file_paths = sorted(glob.glob(os.path.join(dataset_path, "**/*.npz"), recursive=True))

#Create Train/test splits of the file, in the paper 50/50 is used
train_paths, test_val_paths = train_test_split(all_file_paths, test_size = 0.5, random_state = 69, shuffle = True)
val_paths, test_paths = train_test_split(test_val_paths, test_size = 0.5, random_state = 69, shuffle = True)

print(f"Train Paths: {len(train_paths)}", f"Val Paths: {len(val_paths)}", f"Test Paths: {len(test_paths)}")

#Peek at the first few to ensure they look right
for path in all_file_paths[:3]:
    print(path)

In [ ]:
#Helper functions to filter dataset according to session for cross-session training 
def filter_dataset():
    dataset_path = '/kaggle/input/train-preprocessed/data_preprocessed'
    all_file_paths = sorted(glob.glob(os.path.join(dataset_path, "**/*.npz"), recursive=True))
    test_paths=[]
    
    #Dictionary to filter per session:  0 (50), 1 (50), 4 (50), 7 (50), 13 (25), 14 (25)
    #we will merge session 13 and 14 into session 13 to match the number of files from othe sessions
    sessions_data = {0:[], 1:[], 4:[], 7:[], 13:[]}
    
    for path in all_file_paths:
        session_id = int(os.path.basename(path).split('_')[1])  # Extract second digit and turn it into a pin
        if session_id == 14:
            sessions_data[13].append(path) #add session 14 paths into session 13 
        elif session_id in sessions_data:
            sessions_data[session_id].append(path) #fills dictionary if id corresponds to one of the keys
        else:
            test_paths.append(path) #includes all the other sessions in the test set 

    print("Number of files in test_paths: ", len(test_paths))
    # Print counts for verification
    for key, files in sessions_data.items():
        print(f"Session {key} has {len(files)} files")
        
    return sessions_data, test_paths

def create_kfold_dataset(val_session, sessions_data):
    val_paths = sessions_data[val_session] #obtains the paths from the val set
    other_sessions = sessions_data.keys()- {val_session}
        
    #Obtaining training set
    train_paths = []
    for session in other_sessions:
        train_paths.extend(sessions_data[session])
                
    print(f"FOLD: Val Session {val_session}")
    print(f"  Train samples: {len(train_paths)} | Val samples: {len(val_paths)}")
    return train_paths, val_paths

In [ ]:
#Fetch a single RDI to apply the preprocessing to and compare it visually to double check correctness

original_path = '/kaggle/input/solidata/dsp/0_10_17.h5'

original_file = h5py.File(original_path, 'r')
original_seq = np.stack([original_file['ch0'], original_file['ch1'], original_file['ch2'], original_file['ch3']], axis = -1)
original_seq = original_seq.reshape(original_seq.shape[0], 32, 32, original_seq.shape[2])

processed_seq, label = process_soli_file(original_path)

#Double check correct shape
print(processed_seq.shape)

#Visualize log-scaled original RDI (To be able to see the tiny non-zero entries) vs preprocessed
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(np.log1p(original_seq[30, :, :, 0]*100), origin='lower', cmap='viridis')
axes[0].set_title("Original Frame")
axes[1].imshow(processed_seq[30, :, :, 0], origin='lower', cmap='viridis')
axes[1].set_title("Processed Frame")
plt.show()

In [ ]:
#Fetch a single RDI from the preprocessed dataset, and its corresponding original RDI from the
#Original dataset, to make sure datasets are ok.

preprocessed_RDI = load('/kaggle/input/train-preprocessed/data_preprocessed/0_11_1.npz')
original_RDI_path =  '/kaggle/input/solidata/dsp/0_11_1.h5'

original_file = h5py.File(original_RDI_path, 'r')
original_seq = np.stack([original_file['ch0'], original_file['ch1'], original_file['ch2'], original_file['ch3']], axis = -1)
original_seq = original_seq.reshape(original_seq.shape[0], 32, 32, original_seq.shape[2])

print(f"Original File: {original_RDI_path}")
print(f"Preprocessed File: {preprocessed_RDI}")
print(f"X Shape:{preprocessed_RDI['x'].shape}", f"Label: {np.unique(preprocessed_RDI['y'])}")

#Original RDI is log-scaled to bring out visually the near-zero pixels

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(np.log1p(original_seq[30, :, :, 0]*100), origin='lower', cmap='viridis')
axes[0].set_title("Original Frame")
axes[1].imshow(preprocessed_RDI['x'][30, :, :, 0], origin='lower', cmap='viridis')
axes[1].set_title("Processed Frame from folder")
plt.show()

In [ ]:
#Second, create tensorflow dataset objects

cnn_rnn_train_ds = get_cnn_rnn_dataset(train_paths, batch_size = 8, target_frames = 40, norm_type = 'minmax', 
                               trunc_mode = 'random', clip_range = False, clip_amount = 20, shuffle = True)
cnn_rnn_val_ds = get_cnn_rnn_dataset(val_paths, batch_size = 8, target_frames = 40, trunc_mode = 'first',
                             norm_type = 'minmax', clip_range = False, shuffle = False)
cnn_rnn_test_ds = get_cnn_rnn_dataset(test_paths, batch_size = 8, target_frames = 40, trunc_mode = 'first',
                              norm_type = 'minmax', clip_range = False, shuffle = False)

In [ ]:
lsm_train_ds = get_lsm_dataset(train_paths, batch_size = 8, apply_temporal = True, target_frames = 40,
                   trunc_mode = 'middle', threshold = 0, clip_range = False, shuffle = True)

lsm_val_ds = get_lsm_dataset(val_paths, batch_size = 8, apply_temporal = True, target_frames = 40,
                   trunc_mode = 'middle', threshold = 0, clip_range = False, shuffle = False)

lsm_test_ds = get_lsm_dataset(test_paths, batch_size = 8, apply_temporal = True, target_frames = 40,
                   trunc_mode = 'middle', threshold = 0, clip_range = False, shuffle = False)

In [ ]:
#Function to get some statistics on the data, written out of suspicion of vanishing gradients
#After training the model for the first time

def analyse_dataset_stats(dataset, num_batches=20):
    all_means = []
    all_stds = []
    mins = []
    maxs = []
    sample_data = []

    print(f"Analyzing {num_batches} batches for statistics...")
    for i, (images, _) in enumerate(dataset.take(num_batches)):
        img_np = images.numpy()
        # Filter out background (zeros) to see the real signal strength
        active_signal = img_np[img_np > 0]
        
        if active_signal.size > 0:
            all_means.append(np.mean(active_signal))
            all_stds.append(np.std(active_signal))
            mins.append(np.min(active_signal))
            maxs.append(np.max(active_signal))
            # Store a fraction for IQR/Median
            sample_data.extend(np.random.choice(active_signal, size=min(1000, len(active_signal))))

    print("\n--- Radar Signal Statistics (Non-Zero Pixels) ---")
    print(f"Global Min:  {np.min(mins):.8f}")
    print(f"Global Max:  {np.max(maxs):.8f}")
    print(f"Global Mean: {np.mean(all_means):.8f}")
    print(f"Global Std:  {np.mean(all_stds):.8f}")
    
    if sample_data:
        sample_data = np.array(sample_data)
        print(f"Median:      {np.median(sample_data):.8f}")
        print(f"IQR (75-25): {np.percentile(sample_data, 75) - np.percentile(sample_data, 25):.8f}")

print("Statistics for Training dataset")
analyse_dataset_stats(cnn_rnn_train_ds)
print("Statistics for Validation dataset")
analyse_dataset_stats(cnn_rnn_val_ds)
print("Statistics for Testing dataset")
analyse_dataset_stats(cnn_rnn_test_ds)

#Numbers in a really healthy range for a NN, so if there is a vanishing gradient problem it is not
#Rooted in the data itself

In [ ]:
#Check LSM dataset is formed correctly 

#OCCHIO these spikes are not the same as the spike trains formed by passing the input through a reservoir
#These spikes simply refer to the original data binarised, to be appropriate for LSM input.
#LSM operates on a binary basis, i.e. does an input hit a cell or not, rather than the value of the
#input mattering.

for spikes_batch, labels_batch in lsm_train_ds.take(1):
    
    sample_spikes = spikes_batch[0].numpy() 
    sample_label = labels_batch[0].numpy()

    #Check input shape, should be (batch, t 4096)
    print(f"Batch Shape: {spikes_batch.shape}") 

    #Check that spike train only contains 1s and 0s
    unique_values = np.unique(sample_spikes)
    print(f"Unique values in data: {unique_values}") 

    #Check that labels are loading correctly
    unique_labels = np.unique(sample_label)
    print(f"Unique labels in data: {unique_labels}")

In [ ]:
#Check that the CNN RNN dataset objects are working correctly and input is correctly shaped
#Should add two graph sequences for val / test datasets

for images, labels in cnn_rnn_train_ds.take(1):
    print(f"Batch Image Shape: {images.shape}")  # Should be (batch, t, 32, 32, 4)
    print(f"Batch Label Shape: {labels.shape}") # Should be (4,)
    print(f"Labels in this batch: {labels.numpy()}")
    
    # Select the first sequence in the batch
    first_sequence = images[0].numpy() # Shape: (40, 32, 32, 4)
    
    # 3. Visualize a few frames from this sequence
    fig, axes = plt.subplots(1, 5, figsize=(20, 5))
    fig.suptitle("Sequence preview: CNNRNN Training Split")
    
    # Pick 5 frames throughout the sequence (0, 10, 20, 30, 39)
    frame_indices = np.linspace(0, 39, 5, dtype=int)
    
    for i, f_idx in enumerate(frame_indices):
        # Plot only Ch0 for simplicity
        frame_ch0 = first_sequence[f_idx, :, :, 0]
        
        im = axes[i].imshow(frame_ch0, origin='lower', cmap='viridis')
        axes[i].set_title(f"Frame {f_idx}")
        axes[i].axis('off')
        
    plt.show()

#Check that the dataset objects are working correctly and input is correctly shaped
#Should add two graph sequences for val / test datasets

for images, labels in cnn_rnn_val_ds.take(1):
    print(f"Batch Image Shape: {images.shape}")  # Should be (batch, t, 32, 32, 4)
    print(f"Batch Label Shape: {labels.shape}") # Should be (4,)
    print(f"Labels in this batch: {labels.numpy()}")
    
    # Select the first sequence in the batch
    first_sequence = images[0].numpy() # Shape: (40, 32, 32, 4)
    
    # 3. Visualize a few frames from this sequence
    fig, axes = plt.subplots(1, 5, figsize=(20, 5))
    fig.suptitle("Sequence preview: CNNRNN Validation Split")
    
    # Pick 5 frames throughout the sequence (0, 10, 20, 30, 39)
    frame_indices = np.linspace(0, 39, 5, dtype=int)
    
    for i, f_idx in enumerate(frame_indices):
        # Plot only Ch0 for simplicity
        frame_ch0 = first_sequence[f_idx, :, :, 0]
        
        im = axes[i].imshow(frame_ch0, origin='lower', cmap='viridis')
        axes[i].set_title(f"Frame {f_idx}")
        axes[i].axis('off')
        
    plt.show()

#Check that the dataset objects are working correctly and input is correctly shaped
#Should add two graph sequences for val / test datasets

for images, labels in cnn_rnn_test_ds.take(1):
    print(f"Batch Image Shape: {images.shape}")  # Should be (batch, t, 32, 32, 4)
    print(f"Batch Label Shape: {labels.shape}") # Should be (4,)
    print(f"Labels in this batch: {labels.numpy()}")
    
    # Select the first sequence in the batch
    first_sequence = images[0].numpy() # Shape: (40, 32, 32, 4)
    
    # 3. Visualize a few frames from this sequence
    fig, axes = plt.subplots(1, 5, figsize=(20, 5))
    fig.suptitle("Sequence Preview: CNNRNN Test Split")
    
    # Pick 5 frames throughout the sequence (0, 10, 20, 30, 39)
    frame_indices = np.linspace(0, 39, 5, dtype=int)
    
    for i, f_idx in enumerate(frame_indices):
        # Plot only Ch0 for simplicity
        frame_ch0 = first_sequence[f_idx, :, :, 0]
        
        im = axes[i].imshow(frame_ch0, origin='lower', cmap='viridis')
        axes[i].set_title(f"Frame {f_idx}")
        axes[i].axis('off')
        
    plt.show()

#  Model definition

In this section we define the necessary functions and building blocks to put together our model architectures and ease combination and experimentation of different architectures. First we will be imitating the architectures used in Wang2016 and Tsang21:

1. Jointly trained CNN + RNN: Combining the spatial feature extraction of a convolutional neural network to extract features from a stacked input of RDIs, plus the extraction of temporal information inherent to RNNs, using a long-short-term memory cell.

2. Liquid State Machinie: Define a randomly initialized reservoir, sparsely activated (<10% non-zero weights) and sparsely connected (

## CNN + RNN Model

The architecture and tensor flow of this model is as follows:

1. Input: (t, 32, 32, 4) RDI sequence of t frames
2. Convolutional layer 1: (3x3) kernel, 32 filters, padding = same -> (t, 32, 32, 32)
3. MaxPool 2x2, Normalization -> (t, 16, 16, 32)
4. Convolutional layer 2: (3x3) kernel, 64 filters, padding = same -> (t, 16, 16, 64)
5. MaxPool 2x2, Normalization -> (t, 8, 8, 64)
6. Convolutional layer 3: (3x3) kernel, 128 filters, padding = same -> (t, 8, 8, 128)
7. MaxPool 2x2, Normalization -> (t, 4, 4, 128)
8. FC1: 512 neurons, Normalization, ReLu -> (t, 512)
9. FC2: 512 neurons, Normalization, ReLu -> (t, 512)
10. LSTM: 512 cells -> (1, 512)
11. FC3 11 Logit Output Layer: -> (1, 11)

In the paper a more accurate model is described where the pooling layers are removed, but in that case we would have an extremely large amount of parameters, which we do not have the resources to train. In any case, in the paper it is shown that for that parameter increase of more than a factor of 10, the accuracy increase is only a little over 2%.

Let's now analyze the case where we remove the Max Pooling layers from the CNN with zero padding.

1. Input: (t, 32, 32, 4) RDI sequence of t frames
2. Convolutional layer 1: (3x3) kernel, 32 filters, padding = valid -> (t, 30, 30, 32)
3. Convolutional layer 2: (3x3) kernel, 64 filters, padding = valid -> (t, 28, 28, 64), dropout=0.4
4. Convolutional layer 3: (3x3) kernel, 128 filters, padding = valid -> (t, 26, 26, 128), dropout=0.4
5. Flatten: (t, 86.528) 
6. FC1: 512 neurons, Normalization, ReLu -> (t, 512), dropout=0.5
7. FC2: 512 neurons, Normalization, ReLu -> (t, 512), dropout=0.5
8. LSTM: 512 cells -> (1, 512)
9. FC3 11 Logit Output Layer: -> (1, 11)

Following the exact configuration as in the paper, we would obtain 86,528 features, which represent a considerable jump compared to the 2,048 features obtained by using the Max Pooling layer. This can lead to overfitting, so it's important to include the dropout functionality to prevent it.  

Previous model triggers the vanishing gradient problem, so we'll reduce the image size by using bigger filters and stride 2.

Let's now analyze the case where we remove the Max Pooling layers from the CNN with zero padding.

1. Input: (t, 32, 32, 4) RDI sequence of t frames
2. Convolutional layer 1: (4x4) kernel, 32 filters, stride=2, padding = valid -> (t, 15, 15, 32)
3. Convolutional layer 2: (3x3) kernel, 64 filters, stride=2, padding = valid -> (t, 7, 7, 64), dropout=0.4
4. Convolutional layer 3: (4x4) kernel, 128 filters, stride=2, padding = valid -> (t, 4, 4, 128), dropout=0.4
5. Flatten: (t, 2,048) 
6. FC1: 512 neurons, Normalization, ReLu -> (t, 512), dropout=0.5
7. FC2: 512 neurons, Normalization, ReLu -> (t, 512), dropout=0.5
8. LSTM: 512 cells -> (1, 512)
9. FC3 11 Logit Output Layer: -> (1, 11)

In this way, we obtained a model with the same dimensions as our original model, but without the pooling layers. This is a great way to compare the effect of these layers.

In [ ]:
# Define a Convolution function that applies time distributed convolution, normalization and activation

def TDConvBN(X, filters, kernel_size, padding='same', stride=(1,1)):

    X = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Conv2D(filters, kernel_size, padding = padding, strides=stride)
    )(X)
    
    X = tf.keras.layers.TimeDistributed(
        tf.keras.layers.BatchNormalization()
    )(X)

    X = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Activation('relu')
    )(X)

    return X

#Define a Fully Connected layer function that applies time distributed flattening, FC pass, Norm and ReLu  

def TDDenseBN(X, units, activation):

    #Flatten input
    X = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Flatten()
    )(X)

    #Fully Connected pass
    X = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(units)
    )(X)

    #Normalization
    X = tf.keras.layers.TimeDistributed(
        tf.keras.layers.BatchNormalization()
    )(X)

    #Activation
    X = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Activation(activation)
    )(X)

    return X

In [ ]:
#Check that tensor flows correctly through Conv
test_input = tf.random.uniform((1,40,32,32,4))

print(TDConvBN(test_input, 32,(3,3)).shape)


In [ ]:
#Check that tensor flows correctly through Dense layer

test_fc = tf.random.uniform((1,40,4,4,128))

TDDenseBN(test_fc, 512, 'relu').shape

In [ ]:
#Wrap the CNN portion of the model in a block:

def CNN_Block(input_tensor):

    #First Convolution Layer
    X = TDConvBN(input_tensor, 32,(3,3))
    #MaxPool 2x2
    X = tf.keras.layers.TimeDistributed(
        tf.keras.layers.MaxPool2D()
    )(X)
    #Second Convolution Layer
    X = TDConvBN(X, 64, (3,3))
    #MaxPool 2x2
    X = tf.keras.layers.TimeDistributed(
        tf.keras.layers.MaxPool2D()
    )(X)
    #Third Convolution Layer
    X = TDConvBN(X, 128, (3,3))
    #MaxPool 2x2
    X = tf.keras.layers.TimeDistributed(
        tf.keras.layers.MaxPool2D()
    )(X)

    return X
    
def CNN_Block_avgpool(input_tensor):

    #First Convolution Layer
    X = TDConvBN(input_tensor, 32,(3,3))
    #AvgPool 2x2
    X = tf.keras.layers.TimeDistributed(
        tf.keras.layers.AveragePooling2D(pool_size=(2,2))
    )(X)
    #Second Convolution Layer
    X = TDConvBN(X, 64, (3,3))
    #AvgPool 2x2
    X = tf.keras.layers.TimeDistributed(
        tf.keras.layers.AveragePooling2D(pool_size=(2,2))
    )(X)
    #Third Convolution Layer
    X = TDConvBN(X, 128, (3,3))
    #AvgPool 2x2
    X = tf.keras.layers.TimeDistributed(
        tf.keras.layers.AveragePooling2D(pool_size=(2,2))
    )(X)

    return X
    
def CNN_Block_nomaxpool(input_tensor):

    #First Convolution Layer
    X = TDConvBN(input_tensor, 32,(4,4), padding='valid', stride=(2,2))

    #Second Convolution Layer
    X = TDConvBN(X, 64, (3,3), padding='valid', stride=(2,2))
    #X = tf.keras.layers.TimeDistributed(
    #    tf.keras.layers.Dropout(0.4))(X)

    #Third Convolution Layer
    X = TDConvBN(X, 128, (4,4), padding='valid')
    #X = tf.keras.layers.TimeDistributed(
    #    tf.keras.layers.Dropout(0.4))(X)
    
    return X

In [ ]:
#Check tensor flows correctly through CNN with pooling 
CNN_test_output = CNN_Block(test_input)
print(CNN_test_output.shape)

CNN_test_output = CNN_Block_avgpool(test_input)
print(CNN_test_output.shape)

#and CNN without pooling and with dropout
CNN_test_output = CNN_Block_nomaxpool(test_input)
print(CNN_test_output.shape)

In [ ]:
#Wrap the FC layers in a block:

def FC_BN_Block(input_tensor):

    X = TDDenseBN(input_tensor, 512, 'relu')

    X = TDDenseBN(X, 512, 'relu')

    return X

def FC_BN_Block_reduced(input_tensor):

    X = TDDenseBN(input_tensor, 256, 'relu')

    X = TDDenseBN(X, 256, 'relu')

    return X

def FC_BN_Block_dropout(input_tensor):

    X = TDDenseBN(input_tensor, 512, 'relu')
    X = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dropout(0.5))(X)

    X = TDDenseBN(X, 512, 'relu')
    X = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dropout(0.5))(X)

    return X

In [ ]:
#This is a modified version of the autoencoder we used in class. Modifications were applied so we could fit our
#5d input into it 

def build_5d_autoencoder(sequence_shape=(40, 32, 32, 4), code_size=128):
    # --- ENCODER ---
    # input_img shape: (None, 40, 32, 32, 4)
    input_img = tf.keras.layers.Input(shape=sequence_shape)

    # Wrap every CNN layer in TimeDistributed
    x = tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'))(input_img)
    x = tf.keras.layers.TimeDistributed(tf.keras.layers.BatchNormalization())(x)
    x = tf.keras.layers.TimeDistributed(tf.keras.layers.MaxPool2D((2, 2), padding='same'))(x)

    x = tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'))(x)
    x = tf.keras.layers.TimeDistributed(tf.keras.layers.BatchNormalization())(x)
    x = tf.keras.layers.TimeDistributed(tf.keras.layers.MaxPool2D((2, 2), padding='same'))(x)

    x = tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'))(x)
    x = tf.keras.layers.TimeDistributed(tf.keras.layers.BatchNormalization())(x)
    x = tf.keras.layers.TimeDistributed(tf.keras.layers.MaxPool2D((2, 2), padding='same'))(x)

    # Flatten each frame individually into a vector
    x = tf.keras.layers.TimeDistributed(tf.keras.layers.Flatten())(x)
    
    # Compress each frame into the 'code_size'
    # Output shape: (None, 40, code_size)
    code_seq = tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(code_size))(x)

    encoder = Model(input_img, code_seq, name='encoder_5d')

    # --- DECODER ---
    input_code = tf.keras.layers.Input(shape=(40, code_size))

    # Expand each vector back to a 4x4x128 volume
    x = tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(4 * 4 * 128, activation='relu'))(input_code)
    x = tf.keras.layers.TimeDistributed(tf.keras.layers.Reshape((4, 4, 128)))(x)

    # Upsample back to 32x32
    x = tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2DTranspose(64, (3, 3), strides=2, activation='relu', padding='same'))(x)
    x = tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2DTranspose(32, (3, 3), strides=2, activation='relu', padding='same'))(x)
    
    # Final reconstruction: (None, 40, 32, 32, 4)
    reconstruction = tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2DTranspose(4, (3, 3), strides=2, activation='sigmoid', padding='same'))(x)

    decoder = Model(input_code, reconstruction, name='decoder_5d')

    return encoder, decoder


# Combine them
encoder, decoder = build_5d_autoencoder()
autoencoder = Model(encoder.input, decoder(encoder.output))
autoencoder.compile(optimizer='adam', loss='mse')

In [ ]:
encoder.summary()
decoder.summary()

In [ ]:
#We are adjusting our input set, so it considers the image itself as the output. This is necessary to compute the loss
def finalize_dataset_for_5d_ae(ds):
    # We ignore the label 'y' and return (image_batch, image_batch)
    # We use tf.ensure_shape to help the Graph execution understand the dimensions
    return ds.map(lambda x, y: (x, x)).prefetch(tf.data.AUTOTUNE)

ae_train_ds = finalize_dataset_for_5d_ae(cnn_rnn_train_ds)
ae_val_ds = finalize_dataset_for_5d_ae(cnn_rnn_val_ds)

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor = 'val_loss', patience = 6, restore_best_weights=True,
    verbose=1)

batch_size=8

train_steps = int(np.ceil(len(train_paths) / batch_size))
val_steps = int(np.ceil(len(val_paths) / batch_size))

"""
autoencoder.fit(ae_train_ds, epochs=20,
                          steps_per_epoch=train_steps,
                          validation_data=ae_val_ds,
                          validation_steps=val_steps,
                          verbose=1, callbacks=[early_stop])
"""
encoder = tf.keras.models.load_model('/kaggle/input/model-1/tensorflow2/ae_rnn/2/encoder.keras')
encoder.load_weights('/kaggle/input/model-1/tensorflow2/ae_rnn/2/encoder.weights.h5')
encoder.summary()

In [ ]:
def visualize_ae_results(model, dataset, num_sequences=2, frames_to_show=5):
    #Get one batch from the dataset
    for x_batch, _ in dataset.take(1):
        #get model's prediction (reconstruction)
        reconstructions = model.predict(x_batch)
    
        fig, axes = plt.subplots(num_sequences * 2, frames_to_show, figsize=(15, num_sequences * 5))
        
        for i in range(num_sequences):
            # Pick frames at intervals depending on the number of frames to show
            frame_indices = np.linspace(0, 39, frames_to_show, dtype=int)
            
            for j, frame_idx in enumerate(frame_indices):
                # Plot Original (we'll visualize the first channel)
                ax_orig = axes[i*2, j]
                ax_orig.imshow(x_batch[i, frame_idx, :, :, 0], cmap='viridis')
                ax_orig.set_title(f"Seq {i} Frame {frame_idx}\n(Original)")
                ax_orig.axis('off')
                
                #Plot Reconstruction
                ax_recon = axes[i*2 + 1, j]
                ax_recon.imshow(reconstructions[i, frame_idx, :, :, 0], cmap='magma')
                ax_recon.set_title(f"Seq {i} Frame {frame_idx}\n(Reconstructed)")
                ax_recon.axis('off')
        
        plt.tight_layout()
        plt.show()
        break # Only process one batch

#visualize_ae_results(autoencoder, ae_val_ds)

In [ ]:
#Now that our Autoencoder is trained we can "save" it and its weights
# Save and load models
#encoder.save('encoder2.keras')
#encoder = tf.keras.models.load_model('encoder2.keras')

# Save and load only the parameters
#encoder.save_weights('encoder2.weights.h5')
#encoder.load_weights('encoder2.weights.h5')

In [ ]:
#Test tensor flows correctly through FC without dropout
FC_test_output = FC_BN_Block(CNN_test_output)
print(FC_test_output.shape)

#and with dropout
FC_test_output = FC_BN_Block_dropout(CNN_test_output)
print(FC_test_output.shape)

"""
Output dimensions should be the same since the only difference between
them is the dropout function, and this function only "deactivates" the
neurons, but they are still there
"""

In [ ]:
#RNN model head

def RNN_Block(input_tensor, units=512):

    #LSTM with 512 cells
    X = tf.keras.layers.LSTM(units=units)(input_tensor)
    
    #Normalise and activate

    X = tf.keras.layers.BatchNormalization()(X)

    X = tf.keras.layers.Activation('relu')(X)

    #FC with 11 neurons, outputs a logit vector
    X = tf.keras.layers.Dense(units = 11, activation = 'relu')(X)

    return X
    

In [ ]:
#Test tensor flows through correctly

test_logit_output = RNN_Block(FC_test_output)
test_logit_output.shape

In [ ]:
#Wrap everything together to define the model 

RDI_seq_input = tf.keras.Input(shape=(None, 32, 32, 4))
CNN_output = CNN_Block(RDI_seq_input)
FC_output = FC_BN_Block(CNN_output)
logit_output = RNN_Block(FC_output)

CNN_RNN_Model_1 = tf.keras.Model(inputs = RDI_seq_input, outputs = logit_output, name= 'CNN_RNN_Joint_Model_1')

CNN_RNN_Model_1.summary()

In [ ]:
def create_CNN_RNN_nopooling(name):
    RDI_seq_input = tf.keras.Input(shape=(None, 32, 32, 4))
    CNN_output = CNN_Block_nomaxpool(RDI_seq_input)
    FC_output = FC_BN_Block(CNN_output)
    logit_output = RNN_Block(FC_output)
    
    CNN_RNN_Model = tf.keras.Model(inputs = RDI_seq_input, outputs = logit_output, name=name)
    
    CNN_RNN_Model.summary()
    return CNN_RNN_Model

CNN_RNN_Model_4 = create_CNN_RNN_nopooling('CNN_RNN_Joint_Model_4')

In [ ]:
def create_CNN_RNN_avgpooling(name):
    RDI_seq_input = tf.keras.Input(shape=(None, 32, 32, 4))
    CNN_output = CNN_Block_avgpool(RDI_seq_input)
    FC_output = FC_BN_Block(CNN_output)
    logit_output = RNN_Block(FC_output)
    
    CNN_RNN_Model = tf.keras.Model(inputs = RDI_seq_input, outputs = logit_output, name=name)
    
    CNN_RNN_Model.summary()
    return CNN_RNN_Model

CNN_RNN_Model_5 = create_CNN_RNN_avgpooling('CNN_RNN_Joint_Model_5')

In [ ]:
def create_AE_RNN(name, RNN_neurons):
    encoder.trainable=False #we are freezing our model's weights
    RDI_seq_input = tf.keras.Input(shape=(None, 32, 32, 4))
    AE_output = encoder(RDI_seq_input)
    FC_output = FC_BN_Block_reduced(AE_output)
    logit_output = RNN_Block(FC_output, RNN_neurons) #this last number adjusts the number of neurons
    
    AE_RNN_Model = tf.keras.Model(inputs = RDI_seq_input, outputs = logit_output, name=name)
    
    AE_RNN_Model.summary()
    return AE_RNN_Model

AE_RNN_Model_4_CV = create_AE_RNN('AE_RNN_Joint_Model_4_CV', 128)
AE_RNN_Model_5_CV = create_AE_RNN('AE_RNN_Joint_Model_5_CV', 256)  

Now, both models have very similar number of trainable parameters.

* CNN_RNN_Model_1: 3,512,619 (13.40 MB)
* CNN_RNN_Model_4: 3,570,859 (13.62 MB)


So, we expect to reduce the running time of Model 4 and solve the problem of the vanishing gradient. 

## Liquid State Machine Model

For the Liquid State Machine, the LSM itself is a large sparsely connected weight matrix that is not trainable, and keras doesn't have a native 'LSM' layer. However, we can define one using a keras RNN layer with the specific LIF (Leaky Integrate-and-Fire) cell that makes up the LSM reservoir. We want to define it as a tf object so we can more easily integrate it to experimentation of architecture with other blocks, where those are native keras layers. However, for the model imitating the implementation in Tsang21, basically the entire model is not trainable using gradient descent, we can think of the LSM reservoir as a preprocessing function, and we attach scikit-learn classifiers to the end to fit to the data that the LSM output, using optimizers. The architecture therefore is as follows:

1. Input shape (t, HxWxC) of spike trains
2. LSM ( <1000 units) -> (no_of_data, no_units) 'tabular data' of no_units amount of features to feed into classifiers
3. Classifiers (Logistic Regression, Random Forests, SVM) -> Multi-class predictions

The first experiment with a different LSM architecture is to train a model composed of a reservoir, FC-512 layer and a FC-11 logit output layer to see its performance, and then use the trained weights of the FC-512 layer to add on top of a reservoir, to then pass on to a classifier. The two architecures would look as follows:

1. Input shape (t, HxWxC) spike trains
2. LSM ( <1000 units ) -> (1, no_units) 'tabular row' of features for a single input
3. FC-512, normalize, ReLu -> (1, 512) 'readout'
4. FC-11, ReLu -> logit output

Then we can use transfer learning and use a scikit classifier pipeline instead, where we remove the FC-11 from the model above, and use the resulting model to create a tabular readout of the entire dataset to feed into the scikit classifiers:

1. Input shape (t, HxWxC) spike trains
2. Pre-trained model: LSM, FC-512, normalize, ReLu -> (no_of_data, no_of_units) 'tabular data'
3. Classifiers (Logistic Regression, SVM, Random Forests) -> Multi-class predictions 

In [ ]:
#Define the LSM (Leaky Integrate-and-Fire) cell as a Keras layer class to pass as a keras RNN argument

class LIFCell(tf.keras.layers.Layer):
    def __init__(self, units=500, leak_rate=0.2, threshold=1.0, inhibitory_ratio=0.2, sparsity=0.1, c_in = 1.0, **kwargs):
        super(LIFCell, self).__init__(**kwargs)
        self.units = units
        self.state_size = [units, units] # [v_mem, spikes]
        self.leak_rate = leak_rate
        self.threshold = threshold
        self.inhibitory_ratio = inhibitory_ratio
        self.sparsity = sparsity
        self.c_in = c_in

    def get_config(self):
        config = super(LIFCell, self).get_config()
        config.update({
            "units": self.units,
            "leak_rate": self.leak_rate,
            "threshold": self.threshold,
            "inhibitory_ratio": self.inhibitory_ratio,
            "sparsity": self.sparsity,
            "c_in": self.c_in,
        })
        return config

    def build(self, input_shape):
        n_input = int(input_shape[-1])
        n_res = int(self.units)

        w_in_raw = np.random.normal(0, 0.1, (n_input, n_res))
        # Create a mask where each input channel only talks to a fraction of neurons
        in_mask = np.random.rand(n_input, n_res) < self.c_in
        w_in_sparse = w_in_raw * in_mask
        
        # 1. Input Weights (W_in) - Fixed Random
        self.w_in = self.add_weight(shape=(n_input, n_res),
                                    initializer = tf.constant_initializer(w_in_sparse),
                                    trainable=False, name='w_in')

        # 2. Reservoir Weights (W_res) with E/I Balance
        # Initialize raw weights
        w_res_raw = np.random.normal(0, 0.1, (n_res, n_res))
        
        # Determine Inhibitory indices (20%)
        num_inhibitory = int(n_res * self.inhibitory_ratio)
        indices = np.arange(n_res)
        np.random.shuffle(indices)
        inh_indices = indices[:num_inhibitory]
        
        # Apply E/I signs: Excitatory are (+), Inhibitory are (-)
        # Note: Weights coming FROM an inhibitory neuron must be negative
        for i in inh_indices:
            w_res_raw[i, :] = -np.abs(w_res_raw[i, :])
        for i in indices[num_inhibitory:]:
            w_res_raw[i, :] = np.abs(w_res_raw[i, :])
            
        # 3. Apply Sparsity Mask
        mask = np.random.rand(n_res, n_res) < self.sparsity
        w_res_final = w_res_raw * mask
        
        # Set as non-trainable weight
        self.w_res = self.add_weight(shape=(n_res, n_res),
                                     initializer=tf.constant_initializer(w_res_final),
                                     trainable=False, name='w_res')
        self.built = True

    def call(self, inputs, states):
        # states[0]: v_mem (membrane potential)
        # states[1]: prev_spikes (binary spikes from t-1)
        prev_v_mem, prev_spikes = states

        # 1. Integrate: New Input + Recurrent Input
        # (Inputs come from the radar spike encoder)
        synaptic_input = tf.matmul(inputs, self.w_in) + tf.matmul(prev_spikes, self.w_res)
        
        # 2. Leak: Potential decays over time
        v_mem = (1 - self.leak_rate) * prev_v_mem + synaptic_input
        
        # 3. Fire: If v_mem exceeds threshold, spike!
        spikes = tf.cast(tf.greater(v_mem, self.threshold), tf.float32)
        
        # 4. Reset: If a neuron spiked, reset its potential to 0
        new_v_mem = v_mem * (1 - spikes)
        
        return spikes, [new_v_mem, spikes]

In [ ]:
#Create a LSM 'block' function

def LSM_Reservoir(input_tensor, n_reservoir, leak_rate, inhibitory_ratio, sparsity, c_in, seq_bool):

    X = tf.keras.layers.RNN(LIFCell(units = n_reservoir, leak_rate = leak_rate, inhibitory_ratio = inhibitory_ratio, sparsity = sparsity, c_in = c_in), return_sequences = seq_bool)(input_tensor)

    if seq_bool:

        X = tf.keras.layers.GlobalAveragePooling1D()(X)

    return X

In [ ]:
#Check that tensor flows correctly through, spike train on last frame

test_lsm_input = tf.random.uniform((1,40,4096))
test_lsm_output = LSM_Reservoir(test_lsm_input, 1000, 0.2, 0.2, 0.1, 1.0, False)
test_lsm_output.shape

In [ ]:
#Check that tensor flows correctly through, global average of spike train over the sequence

test_lsm_input_2 = tf.random.uniform((1,40,4096))
test_lsm_output_2 = LSM_Reservoir(test_lsm_input, 1000, 0.2, 0.2, 0.1, 1.0, True)
np.unique(test_lsm_output_2)

In [ ]:
#Define a Fully Connected Dense + BatchNorm + ReLu layer function

def DenseBN(input_tensor, units):

    X = tf.keras.layers.Dense(units)(input_tensor)
    X = tf.keras.layers.BatchNormalization()(X)
    X = tf.keras.layers.Activation('relu')(X)

    return X

In [ ]:
#Define the first LSM experimental model as described in the Markdown cell (model 2)
#And an identical model but with GA LSM

LSM_input_2 = tf.keras.Input(shape = (40,4096))
LSM_reservoir_output = LSM_Reservoir(LSM_input_2, 1000, 0.2, 0.2, 0.1, 1.0, False)
LSM_FC_output = tf.keras.layers.Dense(units = 512)(LSM_reservoir_output)
LSM_FC_Norm = tf.keras.layers.BatchNormalization()(LSM_FC_output)
LSM_FC_Act = tf.keras.layers.Activation('relu')(LSM_FC_Norm)
LSM_output_2 = tf.keras.layers.Dense(units = 11)(LSM_FC_Act)

LSM_input_2_GA = tf.keras.Input(shape = (40,4096))
LSM_reservoir_output_GA = LSM_Reservoir(LSM_input_2_GA, 1000, 0.2, 0.2, 0.1, 1.0, True)
LSM_FC_output_GA = tf.keras.layers.Dense(units = 512)(LSM_reservoir_output_GA)
LSM_FC_Norm_GA = tf.keras.layers.BatchNormalization()(LSM_FC_output_GA)
LSM_FC_Act_GA = tf.keras.layers.Activation('relu')(LSM_FC_Norm_GA)
LSM_output_2_GA = tf.keras.layers.Dense(units = 11)(LSM_FC_Act_GA)

LSM_Model_2 = tf.keras.Model(inputs = LSM_input_2, outputs = LSM_output_2, name = 'LSM_Model_2')
LSM_Model_2_GA = tf.keras.Model(inputs = LSM_input_2_GA, outputs = LSM_output_2_GA, name = 'LSM_Model_2_GA')

LSM_Model_2.summary()

In [ ]:
#Define LSM Model 3, identical to above except with optimal LSM configuration
#And a GA version

LSM_input_3 = tf.keras.Input(shape = (40,4096))
LSM_res_output_3 = LSM_Reservoir(LSM_input_3, 256, 0.15, 0.35, 0.05, 1.0, False)
norm_3 = tf.keras.layers.BatchNormalization()(LSM_res_output_3)
act_3 = tf.keras.layers.Activation('relu')(norm_3)
LSM_FC_512 = DenseBN(act_3, 512)
logit_output_3 = tf.keras.layers.Dense(units = 11)(LSM_FC_512)

LSM_input_3_GA = tf.keras.Input(shape = (40,4096))
LSM_res_output_3_GA = LSM_Reservoir(LSM_input_3_GA, 256, 0.15, 0.35, 0.05, 1.0, True)
norm_3_GA = tf.keras.layers.BatchNormalization()(LSM_res_output_3_GA)
act_3_GA = tf.keras.layers.Activation('relu')(norm_3_GA)
LSM_FC_512_GA = DenseBN(act_3_GA, 512)
logit_output_3_GA = tf.keras.layers.Dense(units = 11)(LSM_FC_512_GA)

LSM_Model_3 = tf.keras.Model(inputs = LSM_input_3, outputs = logit_output_3, name = 'LSM_Model_3')
LSM_Model_3_GA = tf.keras.Model(inputs = LSM_input_3_GA, outputs = logit_output_3_GA, name = 'LSM_Model_3_GA')

LSM_Model_3.summary()
LSM_Model_3_GA.summary()

In [ ]:
#Define LSM Model 4, identical to above but FC-256 instead of FC-512
#And a GA version

LSM_input_4 = tf.keras.Input(shape = (40,4096))
LSM_res_output_4 = LSM_Reservoir(LSM_input_4, 256, 0.15, 0.35, 0.05, 1.0, False)
norm_4 = tf.keras.layers.BatchNormalization()(LSM_res_output_4)
act_4 = tf.keras.layers.Activation('relu')(norm_4)
LSM_FC_256_4 = DenseBN(act_4, 256)
logit_output_4 = tf.keras.layers.Dense(units = 11)(LSM_FC_256_4)

LSM_input_4_GA = tf.keras.Input(shape = (40,4096))
LSM_res_output_4_GA = LSM_Reservoir(LSM_input_4_GA, 256, 0.15, 0.35, 0.05, 1.0, True)
norm_4_GA = tf.keras.layers.BatchNormalization()(LSM_res_output_4_GA)
act_4_GA = tf.keras.layers.Activation('relu')(norm_4_GA)
LSM_FC_256_4_GA = DenseBN(act_4_GA, 256)
logit_output_4_GA = tf.keras.layers.Dense(units = 11)(LSM_FC_256_4_GA)

LSM_Model_4 = tf.keras.Model(inputs = LSM_input_4, outputs = logit_output_4, name = 'LSM_Model_4')
LSM_Model_4_GA = tf.keras.Model(inputs = LSM_input_4_GA, outputs = logit_output_4_GA, name = 'LSM_Model_4_GA')

LSM_Model_4.summary()
LSM_Model_4_GA.summary()

In [ ]:
LSM_input_5 = tf.keras.Input(shape = (40,4096))
LSM_res_output_5 = LSM_Reservoir(LSM_input_5, 256, 0.15, 0.35, 0.05, 1.0, False)
norm_5 = tf.keras.layers.BatchNormalization()(LSM_res_output_5)
act_5 = tf.keras.layers.Activation('relu')(norm_5)
LSM_FC_256_5 = DenseBN(act_5, 256)
LSM_FC_128_5 = DenseBN(LSM_FC_256_5, 128)
LSM_FC_64_5 = DenseBN(LSM_FC_128_5, 64)
LSM_FC_32_5 = DenseBN(LSM_FC_64_5, 32)
LSM_FC_16_5 = DenseBN(LSM_FC_32_5, 16)
logit_output_5 = tf.keras.layers.Dense(units = 11)(LSM_FC_16_5)

LSM_input_5_GA = tf.keras.Input(shape = (40,4096))
LSM_res_output_5_GA = LSM_Reservoir(LSM_input_5_GA, 256, 0.15, 0.35, 0.05, 1.0, True)
norm_5_GA = tf.keras.layers.BatchNormalization()(LSM_res_output_5_GA)
act_5_GA = tf.keras.layers.Activation('relu')(norm_5_GA)
LSM_FC_256_5_GA = DenseBN(act_5_GA, 256)
LSM_FC_128_5_GA = DenseBN(LSM_FC_256_5_GA, 128)
LSM_FC_64_5_GA = DenseBN(LSM_FC_128_5_GA, 64)
LSM_FC_32_5_GA = DenseBN(LSM_FC_64_5_GA, 32)
LSM_FC_16_5_GA = DenseBN(LSM_FC_32_5_GA, 16)
logit_output_5_GA = tf.keras.layers.Dense(units = 11)(LSM_FC_16_5_GA)

LSM_Model_5 = tf.keras.Model(inputs = LSM_input_5, outputs = logit_output_5, name = 'LSM_Model_5')
LSM_Model_5_GA = tf.keras.Model(inputs = LSM_input_5_GA, outputs = logit_output_5_GA, name = 'LSM_Model_5_GA')

LSM_Model_5.summary()
LSM_Model_5_GA.summary()

In [ ]:
# Define Experimental Model CNN + LSM 1
# And GA version

CNN_LSM_input_1 = tf.keras.Input(shape = (None, 32, 32, 4))
CNN_output_1 = CNN_Block(CNN_LSM_input_1)
Flat_features = tf.keras.layers.TimeDistributed(
    tf.keras.layers.Flatten()
)(CNN_output_1)
Liquid_output_1 = LSM_Reservoir(Flat_features, 256, 0.15, 0.35, 0.05, 1.0, False)
Liquid_norm_1 = tf.keras.layers.BatchNormalization()(Liquid_output_1)
Liquid_act_1 = tf.keras.layers.Activation('relu')(Liquid_norm_1)
CNN_LSM_logit_1 = tf.keras.layers.Dense(units = 11)(Liquid_act_1)

CNN_LSM_input_1_GA = tf.keras.Input(shape = (None, 32, 32, 4))
CNN_output_1_GA = CNN_Block(CNN_LSM_input_1_GA)
Flat_features_GA = tf.keras.layers.TimeDistributed(
    tf.keras.layers.Flatten()
)(CNN_output_1_GA)
Liquid_output_1_GA = LSM_Reservoir(Flat_features_GA, 256, 0.15, 0.35, 0.05, 1.0, True)
Liquid_norm_1_GA = tf.keras.layers.BatchNormalization()(Liquid_output_1_GA)
Liquid_act_1_GA = tf.keras.layers.Activation('relu')(Liquid_norm_1_GA)
CNN_LSM_logit_1_GA = tf.keras.layers.Dense(units = 11)(Liquid_act_1_GA)

CNN_LSM_Model_1 = tf.keras.Model(inputs = CNN_LSM_input_1, outputs = CNN_LSM_logit_1, name = 'CNN_LSM_Model_1')
CNN_LSM_Model_1_GA = tf.keras.Model(inputs = CNN_LSM_input_1_GA, outputs = CNN_LSM_logit_1_GA, name = 'CNN_LSM_Model_1_GA')

CNN_LSM_Model_1.summary()
CNN_LSM_Model_1_GA.summary()

In [ ]:
#Define experimental CNN + LSM model 2
#And GA version

CNN_LSM_input_2 = tf.keras.Input(shape = (None, 32, 32, 4))
CNN_output_2 = CNN_Block(CNN_LSM_input_2)
Flat_features_2 = tf.keras.layers.TimeDistributed(
    tf.keras.layers.Flatten()
)(CNN_output_2)
Dense_output_2 = TDDenseBN(Flat_features_2, units = 4096, activation = 'relu')
Liquid_output_2 = LSM_Reservoir(Dense_output_2, 256, 0.15, 0.35, 0.05, 1.0, False)
Liquid_norm_2 = tf.keras.layers.BatchNormalization()(Liquid_output_2)
Liquid_act_2 = tf.keras.layers.Activation('relu')(Liquid_norm_2)
CNN_LSM_logit_2 = tf.keras.layers.Dense(units = 11)(Liquid_act_2)

CNN_LSM_input_2_GA = tf.keras.Input(shape = (None, 32, 32, 4))
CNN_output_2_GA = CNN_Block(CNN_LSM_input_2_GA)
Flat_features_2_GA = tf.keras.layers.TimeDistributed(
    tf.keras.layers.Flatten()
)(CNN_output_2_GA)
Dense_output_2_GA = TDDenseBN(Flat_features_2_GA, units = 4096, activation = 'relu')
Liquid_output_2_GA = LSM_Reservoir(Dense_output_2_GA, 256, 0.15, 0.35, 0.05, 1.0, True)
Liquid_norm_2_GA = tf.keras.layers.BatchNormalization()(Liquid_output_2_GA)
Liquid_act_2_GA = tf.keras.layers.Activation('relu')(Liquid_norm_2_GA)
CNN_LSM_logit_2_GA = tf.keras.layers.Dense(units = 11)(Liquid_act_2_GA)

CNN_LSM_Model_2 = tf.keras.Model(inputs = CNN_LSM_input_2, outputs = CNN_LSM_logit_2, name = 'CNN_LSM_Model_2')
CNN_LSM_Model_2_GA = tf.keras.Model(inputs = CNN_LSM_input_2_GA, outputs = CNN_LSM_logit_2_GA, name = 'CNN_LSM_Model_2_GA')

CNN_LSM_Model_2.summary()
CNN_LSM_Model_2_GA.summary()

In [ ]:
#Create an AE + LSM model

#encoder = tf.keras.models.load_model('/kaggle/input/models/xxdiegoalonsoxx/model-1/tensorflow2/ae_rnn/2/encoder.keras')
#encoder.load_weights('/kaggle/input/models/xxdiegoalonsoxx/model-1/tensorflow2/ae_rnn/2/encoder.weights.h5')

encoder.trainable = False

AE_LSM_input_1 = tf.keras.Input(shape=(None, 32, 32, 4))
Encoder_output_1 = encoder(AE_LSM_input_1)
Liquid_output_AELSM_1 = LSM_Reservoir(Encoder_output_1, 256, 0.15, 0.35, 0.05, 1.0, True)
Liquid_norm_AELSM_1 = tf.keras.layers.BatchNormalization()(Liquid_output_AELSM_1)
Liquid_act_AELSM_1 = tf.keras.layers.Activation('relu')(Liquid_norm_AELSM_1)
AE_LSM_logit_1 = tf.keras.layers.Dense(units = 11)(Liquid_act_AELSM_1)

AE_LSM_Model_1 = tf.keras.Model(inputs = AE_LSM_input_1, outputs = AE_LSM_logit_1, name = 'AE_LSM_Model_1')

AE_LSM_Model_1.summary()

In [ ]:
#encoder = tf.keras.models.load_model('/kaggle/input/models/xxdiegoalonsoxx/model-1/tensorflow2/ae_rnn/2/encoder.keras')
#encoder.load_weights('/kaggle/input/models/xxdiegoalonsoxx/model-1/tensorflow2/ae_rnn/2/encoder.weights.h5')

encoder.trainable = False

AE_LSM_input_2 = tf.keras.Input(shape=(None, 32, 32, 4))
Encoder_output_2 = encoder(AE_LSM_input_2)
Liquid_output_AELSM_2 = LSM_Reservoir(Encoder_output_2, 256, 0.05, 0.3, 0.15, 0.4, True)
Liquid_norm_AELSM_2 = tf.keras.layers.BatchNormalization()(Liquid_output_AELSM_2)
Liquid_act_AELSM_2 = tf.keras.layers.Activation('relu')(Liquid_norm_AELSM_2)
AE_LSM_logit_2 = tf.keras.layers.Dense(units = 11)(Liquid_act_AELSM_2)

AE_LSM_Model_2 = tf.keras.Model(inputs = AE_LSM_input_2, outputs = AE_LSM_logit_2, name = 'AE_LSM_Model_2')

AE_LSM_Model_2.summary()

In [ ]:
#encoder = tf.keras.models.load_model('/kaggle/input/models/xxdiegoalonsoxx/model-1/tensorflow2/ae_rnn/2/encoder.keras')
#encoder.load_weights('/kaggle/input/models/xxdiegoalonsoxx/model-1/tensorflow2/ae_rnn/2/encoder.weights.h5')

encoder.trainable = False

AE_LSM_input_3 = tf.keras.Input(shape=(None, 32, 32, 4))
Encoder_output_3 = encoder(AE_LSM_input_3)
FC_output_3 = TDDenseBN(X = Encoder_output_3, units = 64, activation = 'relu')
Liquid_output_AELSM_3 = LSM_Reservoir(FC_output_3, 256, 0.05, 0.3, 0.15, 0.4, True)
Liquid_norm_AELSM_3 = tf.keras.layers.BatchNormalization()(Liquid_output_AELSM_3)
Liquid_act_AELSM_3 = tf.keras.layers.Activation('relu')(Liquid_norm_AELSM_3)
AE_LSM_logit_3 = tf.keras.layers.Dense(units = 11)(Liquid_act_AELSM_3)

AE_LSM_Model_3 = tf.keras.Model(inputs = AE_LSM_input_3, outputs = AE_LSM_logit_3, name = 'AE_LSM_Model_3')

AE_LSM_Model_3.summary()

# Model Training

In this section we define necessary metrics, callbacks and other tools to be used during the training and validation stages for both models.

In [ ]:
#Loss and metric objects

loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits = True)
metric_SPCC = tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')

#Callbacks

#Learning rate scheduler callback
def poly_decay(epoch):
    
    initial_lrate = 3e-4
    max_epochs = 50
    power = 0.9

    lrate = initial_lrate * (1 - (epoch / max_epochs))**power

    print(f" - Epoch {epoch + 1}: Learning Rate is {lrate:.8f}")

    return lrate

lr_callback = tf.keras.callbacks.LearningRateScheduler(poly_decay, verbose=1)

#Reduce learning rate every 20 epochs by 
def lr_step_decay(epoch):
    initial_lr = 0.001  # 10^-3
    drop_rate = 0.1     # Decrease by a tenth
    epochs_drop = 20.0  # Every 20 epochs
    
    # Calculate the new learning rate
    lr = initial_lr * math.pow(drop_rate, math.floor(epoch / epochs_drop))
    return lr

lr_callback_2 = tf.keras.callbacks.LearningRateScheduler(lr_step_decay, verbose=1)

#Gradient monitor callback to try and diagnose the reason for model accuracy plateau in training
#As we saw that if there is vanishing gradients the cause is not the data itself

class GradientMonitor(tf.keras.callbacks.Callback):
    def __init__(self, dataset):
        
        super().__init__()
        self.sample_images, self.sample_labels = next(iter(dataset.take(1)))
        
    def on_epoch_end(self, epoch, logs=None):
        
        with tf.GradientTape() as tape:
            preds = self.model(self.sample_images, training=True)
            loss = self.model.compiled_loss(self.sample_labels, preds)
            
        # Calculate gradients for all trainable weights
        grads = tape.gradient(loss, self.model.trainable_weights)
        
        # Calculate the global norm (magnitude of all gradients combined)
        # Filter out None gradients (if any)
        valid_grads = [g for g in grads if g is not None]
        global_norm = tf.linalg.global_norm(valid_grads).numpy()
        
        print(f"\n[Epoch {epoch+1}] Global Gradient Norm: {global_norm:.6f}")
        
        if global_norm < 1e-7:
            print("WARNING: Vanishing Gradients detected. Norm is near zero.")
        elif global_norm > 100:
            print("WARNING: Exploding Gradients detected. Consider gradient clipping.")

#Function to plot accuracy and loss on real time during training 
class RealTimePlot(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.epochs = []
        self.accuracy = []
        self.val_accuracy = []
        self.loss = []
        self.val_loss = []

    def on_epoch_end(self, epoch, logs=None):
        # Update metrics - Use .get() to avoid KeyError if names are slightly different
        self.epochs.append(epoch)
        self.accuracy.append(logs.get('accuracy'))
        self.val_accuracy.append(logs.get('val_accuracy'))
        self.loss.append(logs.get('loss'))
        self.val_loss.append(logs.get('val_loss'))
        
        # Plotting
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5))
        
        # Accuracy Plot
        ax1.plot(self.epochs, self.accuracy, label='Train Acc')
        ax1.plot(self.epochs, self.val_accuracy, label='Val Acc')
        ax1.set_title(f'Accuracy (Epoch {epoch+1})')
        ax1.legend()

        # Loss Plot
        ax2.plot(self.epochs, self.loss, label='Train Loss')
        ax2.plot(self.epochs, self.val_loss, label='Val Loss')
        ax2.set_title(f'Loss (Epoch {epoch+1})')
        ax2.legend()

        plt.tight_layout() #prevent the labels from overlapping
        plt.show()

rt_callback = RealTimePlot()

#Early stop
early_stop_callback = tf.keras.callbacks.EarlyStopping(monitor = 'val_loss', patience = 6, restore_best_weights=True,
    verbose=1)

In [ ]:
def Compiler(model, lr=3e-4):
    model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate = lr),
    loss = loss_object,
    metrics = [metric_SPCC]
)

In [ ]:
#Define a function to generate the 'tabular' data readout for the scikit classifiers, with x and y
#From a predefined LSM 'model'

def LSM_Readout(model, dataset):
    features = []
    labels = []
    for x_batch, y_batch in dataset:
        readout = model.predict(x_batch, verbose = 0)
        features.append(readout)
        labels.append(y_batch.numpy())
    return np.vstack(features), np.concatenate(labels)

In [ ]:
# #Perform search for an optimal amount of cells in the liquid, testing by fitting the different
# #Scikit classifiers and scoring them to the readout of the different liquids

# LSM_input_gen = tf.keras.Input(shape = (40,4096))

# svm_accuracies = []
# rf_accuracies = []
# lr_accuracies = []

# n = [int(x) for x in np.linspace(100,600,11)]

# for i in n:
    
#     LSM_output = LSM_Reservoir(LSM_input_gen, i, 0.35, 0.15, 0.05, 1.0, True)
#     LSM_Model_gen = tf.keras.Model(inputs = LSM_input_gen, outputs = LSM_output, name = 'LSM_Model_test')

#     X_train, Y_train = LSM_Readout(LSM_Model_gen, lsm_train_ds)
#     X_val, Y_val = LSM_Readout(LSM_Model_gen, lsm_val_ds)

#     svm = SVC(C = 128)
#     svm.fit(X_train, Y_train)

#     rf = RandomForestClassifier(n_estimators = 200, criterion = 'entropy')
#     rf.fit(X_train, Y_train)

#     lr = LogisticRegression(penalty = 'l2', max_iter = 5000)
#     lr.fit(X_train, Y_train)

#     svm_accuracy = svm.score(X_val, Y_val)
#     rf_accuracy = rf.score(X_val, Y_val)
#     lr_accuracy = lr.score(X_val, Y_val)

#     svm_accuracies.append(svm_accuracy)
#     rf_accuracies.append(rf_accuracy)
#     lr_accuracies.append(lr_accuracy)

#     del LSM_output
#     del LSM_Model_gen

In [ ]:
# #Perform search for an optimal leak_rate in the liquid, testing by fitting the different
# #Scikit classifiers and scoring them to the readout of the different liquids
# #Setting n = 256

# LSM_input_gen = tf.keras.Input(shape = (40,4096))

# svm_accuracies_2 = []
# rf_accuracies_2 = []
# lr_accuracies_2 = []

# sparsity = [x for x in np.linspace(0.06,0.6,11)]

# for i in sparsity:
    
#     LSM_output = LSM_Reservoir(LSM_input_gen, 256, i, 0.15, 0.05, 1.0, True)
#     LSM_Model_gen = tf.keras.Model(inputs = LSM_input_gen, outputs = LSM_output, name = 'LSM_Model_test')

#     X_train, Y_train = LSM_Readout(LSM_Model_gen, lsm_train_ds)
#     X_val, Y_val = LSM_Readout(LSM_Model_gen, lsm_val_ds)

#     svm = SVC(C = 128)
#     svm.fit(X_train, Y_train)

#     rf = RandomForestClassifier(n_estimators = 200, criterion = 'entropy')
#     rf.fit(X_train, Y_train)

#     lr = LogisticRegression(penalty = 'l2', max_iter = 5000)
#     lr.fit(X_train, Y_train)

#     svm_accuracy = svm.score(X_val, Y_val)
#     rf_accuracy = rf.score(X_val, Y_val)
#     lr_accuracy = lr.score(X_val, Y_val)

#     svm_accuracies_2.append(svm_accuracy)
#     rf_accuracies_2.append(rf_accuracy)
#     lr_accuracies_2.append(lr_accuracy)

#     del LSM_output
#     del LSM_Model_gen

In [ ]:
# #Perform search for an optimal inhibitory ratio in the liquid, testing by fitting the different
# #Scikit classifiers and scoring them to the readout of the different liquids
# #Setting n = 256
# #Setting sparsity = 0.05 

# LSM_input_gen = tf.keras.Input(shape = (40,4096))

# svm_accuracies_3 = []
# rf_accuracies_3 = []
# lr_accuracies_3 = []

# inhibitory = [x for x in np.linspace(0.01,0.51,11)]

# for i in inhibitory:
    
#     LSM_output = LSM_Reservoir(LSM_input_gen, 256, 0.05, i, 0.05, 1.0, True)
#     LSM_Model_gen = tf.keras.Model(inputs = LSM_input_gen, outputs = LSM_output, name = 'LSM_Model_test')

#     X_train, Y_train = LSM_Readout(LSM_Model_gen, lsm_train_ds)
#     X_val, Y_val = LSM_Readout(LSM_Model_gen, lsm_val_ds)

#     svm = SVC(C = 128)
#     svm.fit(X_train, Y_train)

#     rf = RandomForestClassifier(n_estimators = 200, criterion = 'entropy')
#     rf.fit(X_train, Y_train)

#     lr = LogisticRegression(penalty = 'l2', max_iter = 5000)
#     lr.fit(X_train, Y_train)

#     svm_accuracy = svm.score(X_val, Y_val)
#     rf_accuracy = rf.score(X_val, Y_val)
#     lr_accuracy = lr.score(X_val, Y_val)

#     svm_accuracies_3.append(svm_accuracy)
#     rf_accuracies_3.append(rf_accuracy)
#     lr_accuracies_3.append(lr_accuracy)

#     del LSM_output
#     del LSM_Model_gen

In [ ]:
# #Perform search for optimal sparsity in the liquid, testing by fitting the different
# #Scikit classifiers and scoring them to the readout of the different liquids
# # Setting n = 256
# # Setting leak rate = 5% 
# # Setting inhibitory ratio =  30% 

# LSM_input_gen = tf.keras.Input(shape = (40,4096))

# svm_accuracies_4 = []
# rf_accuracies_4 = []
# lr_accuracies_4 = []

# sparsity_actual = [x for x in np.linspace(0.01,0.21,11)]

# for i in sparsity_actual:
    
#     LSM_output = LSM_Reservoir(LSM_input_gen, 256, 0.05, 0.3, i, 1.0, True)
#     LSM_Model_gen = tf.keras.Model(inputs = LSM_input_gen, outputs = LSM_output, name = 'LSM_Model_test')

#     X_train, Y_train = LSM_Readout(LSM_Model_gen, lsm_train_ds)
#     X_val, Y_val = LSM_Readout(LSM_Model_gen, lsm_val_ds)

#     svm = SVC(C = 128)
#     svm.fit(X_train, Y_train)

#     rf = RandomForestClassifier(n_estimators = 200, criterion = 'entropy')
#     rf.fit(X_train, Y_train)

#     lr = LogisticRegression(penalty = 'l2', max_iter = 5000)
#     lr.fit(X_train, Y_train)

#     svm_accuracy = svm.score(X_val, Y_val)
#     rf_accuracy = rf.score(X_val, Y_val)
#     lr_accuracy = lr.score(X_val, Y_val)

#     svm_accuracies_4.append(svm_accuracy)
#     rf_accuracies_4.append(rf_accuracy)
#     lr_accuracies_4.append(lr_accuracy)

#     del LSM_output
#     del LSM_Model_gen

In [ ]:
# #Perform search for an optimal c_in sparsity in the liquid, testing by fitting the different
# #Scikit classifiers and scoring them to the readout of the different liquids
# # Setting n = 256
# # Setting leak rate = 5%
# # Setting inhibitory ratio = 30%
# # Setting sparsity = 15%

# LSM_input_gen = tf.keras.Input(shape = (40,4096))

# svm_accuracies_5 = []
# rf_accuracies_5 = []
# lr_accuracies_5 = []

# c_in = [x for x in np.linspace(0.01, 1.0 ,11)]

# for i in c_in:
    
#     LSM_output = LSM_Reservoir(LSM_input_gen, 256, 0.05, 0.3, 0.15, i, True)
#     LSM_Model_gen = tf.keras.Model(inputs = LSM_input_gen, outputs = LSM_output, name = 'LSM_Model_test')

#     X_train, Y_train = LSM_Readout(LSM_Model_gen, lsm_train_ds)
#     X_val, Y_val = LSM_Readout(LSM_Model_gen, lsm_val_ds)

#     svm = SVC(C = 128)
#     svm.fit(X_train, Y_train)

#     rf = RandomForestClassifier(n_estimators = 200, criterion = 'entropy')
#     rf.fit(X_train, Y_train)

#     lr = LogisticRegression(penalty = 'l2', max_iter = 5000)
#     lr.fit(X_train, Y_train)

#     svm_accuracy = svm.score(X_val, Y_val)
#     rf_accuracy = rf.score(X_val, Y_val)
#     lr_accuracy = lr.score(X_val, Y_val)

#     svm_accuracies_4.append(svm_accuracy)
#     rf_accuracies_4.append(rf_accuracy)
#     lr_accuracies_4.append(lr_accuracy)

#     del LSM_output
#     del LSM_Model_gen

In [ ]:
# fig, axs = plt.subplots(1,3)
# fig.suptitle('N vs accuracy for Liquid')

# fig.set_figheight(5)
# fig.set_figwidth(15)

# axs[0].plot(n, svm_accuracies)
# axs[0].set_title('SVM')
# axs[1].plot(n, rf_accuracies)
# axs[1].set_title('Random Forest')
# axs[2].plot(n, lr_accuracies)
# axs[2].set_title('Logistic Regression')

# for ax in axs.flat:
#     ax.set(xlabel = 'N', ylabel = 'Accuracy')

# for ax in axs.flat:
#     ax.label_outer()

In [ ]:
# #actually leak rate

# leak = [x for x in np.linspace(0.06, 0.6, 11)]

# fig, axs = plt.subplots(1,3)
# fig.suptitle('N = 256, Leak Rate vs Accuracy for Liquid')

# fig.set_figheight(5)
# fig.set_figwidth(15)

# axs[0].plot(leak, svm_accuracies_2)
# axs[0].set_title('SVM')
# axs[1].plot(leak, rf_accuracies_2)
# axs[1].set_title('Random Forest')
# axs[2].plot(leak, lr_accuracies_2)
# axs[2].set_title('Logistic Regression')

# for ax in axs.flat:
#     ax.set(xlabel = 'Leak Rate', ylabel = 'Accuracy')

# for ax in axs.flat:
#     ax.label_outer()

In [ ]:
# fig, axs = plt.subplots(1,3)
# fig.suptitle('N = 256, Leak Rate = 0.05%, Inhibitory Ratio vs Accuracy for Liquid')

# fig.set_figheight(5)
# fig.set_figwidth(15)

# axs[0].plot(inhibitory, svm_accuracies_3)
# axs[0].set_title('SVM')
# axs[1].plot(inhibitory, rf_accuracies_3)
# axs[1].set_title('Random Forest')
# axs[2].plot(inhibitory, lr_accuracies_3)
# axs[2].set_title('Logistic Regression')

# for ax in axs.flat:
#     ax.set(xlabel = 'Inhibitory Ratio', ylabel = 'Accuracy')

# for ax in axs.flat:
#     ax.label_outer()

In [ ]:
# fig, axs = plt.subplots(1,3)
# fig.suptitle('N = 256, Sparsity = 5%, Inhibitory Ratio = 30%, Sparsity vs Accuracy for Liquid')

# fig.set_figheight(5)
# fig.set_figwidth(15)

# axs[0].plot(sparsity_actual, svm_accuracies_4)
# axs[0].set_title('SVM')
# axs[1].plot(sparsity_actual, rf_accuracies_4)
# axs[1].set_title('Random Forest')
# axs[2].plot(sparsity_actual, lr_accuracies_4)
# axs[2].set_title('Logistic Regression')

# for ax in axs.flat:
#     ax.set(xlabel = 'Sparsity', ylabel = 'Accuracy')

# for ax in axs.flat:
#     ax.label_outer()

In [ ]:
# fig, axs = plt.subplots(1,3)
# fig.suptitle('N = 250, Leak = 5%, Inhibitory Ratio = 30%, Sparsity = 15%, C_in vs Accuracy')

# fig.set_figheight(5)
# fig.set_figwidth(15)

# axs[0].plot(c_in, svm_accuracies_4[12:23])
# axs[0].set_title('SVM')
# axs[1].plot(c_in, rf_accuracies_4[12:23])
# axs[1].set_title('Random Forest')
# axs[2].plot(c_in, lr_accuracies_4[12:23])
# axs[2].set_title('Logistic Regression')

# for ax in axs.flat:
#     ax.set(xlabel = 'C_in', ylabel = 'Accuracy')

# for ax in axs.flat:
#     ax.label_outer()

In [ ]:
#Define a keras model using the LSM block, even though it has 0 trainable parameters
#To use it to output a readout to feed into the scikit classifiers

LSM_input_paper = tf.keras.Input(shape = (40,4096))
LSM_output_paper = LSM_Reservoir(LSM_input_paper, 1000, 0.2, 0.2, 0.1, False, 1.0)

LSM_input_opt = tf.keras.Input(shape = (40, 4096))
LSM_output_opt = LSM_Reservoir(LSM_input_opt, 256, 0.15, 0.35, 0.05, False, 1.0)

LSM_input_opt_2 = tf.keras.Input(shape = (40,4096))
LSM_output_opt_2 = LSM_Reservoir(LSM_input_opt_2, 256, 0.05, 0.3, 0.15, True, 0.4)

LSM_Model_paper = tf.keras.Model(inputs = LSM_input_paper, outputs = LSM_output_paper, name = 'LSM_Model_paper')
LSM_Model_opt = tf.keras.Model(inputs = LSM_input_opt, outputs = LSM_output_opt, name = 'LSM_Model_opt')
LSM_Model_opt_2 = tf.keras.Model(inputs = LSM_input_opt_2, outputs = LSM_output_opt_2, name = 'LSM_Model_opt_2')

LSM_Model_paper.summary()
LSM_Model_opt.summary()
LSM_Model_opt_2.summary()

In [ ]:
#Create a model using the whole-sequence Global Average LSM:

LSM_input_GA= tf.keras.Input(shape = (40, 4096))
LSM_output_GA = LSM_Reservoir(LSM_input_GA, 256, 0.15, 0.35, 0.05, 1.0, True)

LSM_Model_GA = tf.keras.Model(inputs = LSM_input_GA, outputs = LSM_output_GA, name = 'LSM_Model_GA')
LSM_Model_GA.summary()

In [ ]:
#Part of the training 
# #Generate two readouts: One as per the described parameters in the paper
# # And the other one with the best parameters found above

# X_train_paper, Y_train_paper = LSM_Readout(LSM_Model_paper, lsm_train_ds)
# X_val_paper, Y_val_paper = LSM_Readout(LSM_Model_paper, lsm_val_ds)

# X_train_opt, Y_train_opt = LSM_Readout(LSM_Model_opt, lsm_train_ds)
# X_val_opt, Y_val_opt = LSM_Readout(LSM_Model_opt, lsm_val_ds)

In [ ]:
# #Readout for opt model 2

# X_train_opt_2, Y_train_opt_2 = LSM_Readout(LSM_Model_opt_2, lsm_train_ds)
# X_val_opt_2, Y_val_opt_2 = LSM_Readout(LSM_Model_opt_2, lsm_val_ds)

In [ ]:
# LSM_Model_GA.save('LSM_Model_Readout.keras')

In [ ]:
# #Generate Readouts using the GA LSM:

# X_train_GA, Y_train_GA = LSM_Readout(LSM_Model_GA, lsm_train_ds)
# X_val_GA, Y_val_GA = LSM_Readout(LSM_Model_GA, lsm_val_ds)

In [ ]:
# #Define the LSM readout classifiers, fit them and evaluate them
# #This is LSM 'model 1'

# svm_paper = SVC(C = 128)
# svm_paper.fit(X_train_paper, Y_train_paper)

# rf_paper = RandomForestClassifier(n_estimators = 200, criterion = 'entropy')
# rf_paper.fit(X_train_paper, Y_train_paper)

# lr_paper = LogisticRegression(penalty = 'l2', max_iter = 5000)
# lr_paper.fit(X_train_paper, Y_train_paper)

# svm_opt = SVC(C = 128)
# svm_opt.fit(X_train_opt, Y_train_opt)

# rf_opt = RandomForestClassifier(n_estimators = 200, criterion = 'entropy')
# rf_opt.fit(X_train_opt, Y_train_opt)

# lr_opt = LogisticRegression(penalty = 'l2', max_iter = 5000)
# lr_opt.fit(X_train_opt, Y_train_opt)

# svm_GA = SVC(C = 128)
# svm_GA.fit(X_train_GA, Y_train_GA)

# rf_GA = RandomForestClassifier(n_estimators = 200, criterion = 'entropy')
# rf_GA.fit(X_train_GA, Y_train_GA)

# lr_GA = LogisticRegression(penalty = 'l2', max_iter = 5000)
# lr_GA.fit(X_train_GA, Y_train_GA)

# print(f"SVM Paper Accuracy: {svm_paper.score(X_val_paper, Y_val_paper):.4f}")
# print(f"Random Forest Paper Accuracy: {rf_paper.score(X_val_paper, Y_val_paper):.4f}")
# print(f"Logistic Regression Paper Accuracy: {lr_paper.score(X_val_paper, Y_val_paper):.4f}")

# print(f"SVM Opt Parameters Accuracy: {svm_opt.score(X_val_opt, Y_val_opt):.4f}")
# print(f"Random Forest Opt Parameters Accuracy: {rf_opt.score(X_val_opt, Y_val_opt):.4f}")
# print(f"Logistic Regression Opt Parameters Accuracy: {lr_opt.score(X_val_opt, Y_val_opt):.4f}")

# print(f"SVM GA Parameters Accuracy: {svm_GA.score(X_val_GA, Y_val_GA):.4f}")
# print(f"Random Forest GA Parameters Accuracy: {rf_GA.score(X_val_GA, Y_val_GA):.4f}")
# print(f"Logistic Regression GA Parameters Accuracy: {lr_GA.score(X_val_GA, Y_val_GA):.4f}")

In [ ]:
# #Score model opt 2

# svm_O2 = SVC(C = 128)
# svm_O2.fit(X_train_opt_2, Y_train_opt_2)

# rf_O2 = RandomForestClassifier(n_estimators = 200, criterion = 'entropy')
# rf_O2.fit(X_train_opt_2, Y_train_opt_2)

# lr_O2 = LogisticRegression(penalty = 'l2', max_iter = 5000)
# lr_O2.fit(X_train_opt_2, Y_train_opt_2)

# print(f"SVM GA Parameters Accuracy: {svm_O2.score(X_val_opt_2, Y_val_opt_2):.4f}")
# print(f"Random Forest GA Parameters Accuracy: {rf_O2.score(X_val_opt_2, Y_val_opt_2):.4f}")
# print(f"Logistic Regression GA Parameters Accuracy: {lr_O2.score(X_val_opt_2, Y_val_opt_2):.4f}")

## Cross-validation training

In [ ]:
def cross_session(model1, model2, name1, name2):
    sessions_data, test_paths = filter_dataset() #we filter the data according to the sessions made by the same user
    #The test will remain untouched during the training so we can obtain the dataset already
    cnn_rnn_test_ds_cv = get_cnn_rnn_dataset(test_paths, batch_size = 8, target_frames = 40, trunc_mode = 'first',
                              norm_type = 'minmax', clip_range = False, shuffle = False)
    
    sessions=[0,1,4,7,13] #we exlude the 14 since sessions 13 and 14 are already merged into the 13 
    all_histories1={}
    all_histories2={}
    
    #For to change the validation session every time 
    for val_session in sessions:
        train_paths, val_paths = create_kfold_dataset(val_session, sessions_data) #obtain the train and val paths 
        cnn_rnn_train_ds = get_cnn_rnn_dataset(train_paths, batch_size = 8, target_frames = 40, norm_type = 'minmax', 
                               trunc_mode = 'random', clip_range = False, clip_amount = 20, shuffle = True)
        cnn_rnn_val_ds = get_cnn_rnn_dataset(val_paths, batch_size = 8, target_frames = 40, trunc_mode = 'first',
                             norm_type = 'minmax', clip_range = False, shuffle = False)
        Compiler(model1)
        Compiler(model2)
        
        history1 = model1.fit(cnn_rnn_train_ds, 
                           epochs = 10,
                           validation_data = cnn_rnn_val_ds, 
                           callbacks = [lr_callback, GradientMonitor(cnn_rnn_train_ds), rt_callback], 
                           verbose = 1)
        all_histories1[val_session]=history1.history

        history2 = model2.fit(cnn_rnn_train_ds, 
                           epochs = 10,
                           validation_data = cnn_rnn_val_ds, 
                           callbacks = [lr_callback, GradientMonitor(cnn_rnn_train_ds), rt_callback], 
                           verbose = 1)
        all_histories2[val_session]=history2.history
        
    model1.save(name1)
    model2.save(name2)


    return model1, model2, cnn_rnn_test_ds_cv, all_histories1, all_histories2

In [ ]:
"""
AE_RNN_Model_4_CV, AE_RNN_Model_5_CV, cnn_rnn_test_ds_cv, all_histories1, all_histories2 = cross_session(AE_RNN_Model_4_CV, 
                                                                                            AE_RNN_Model_5_CV,
                                                                                            name1='AE_RNN_Model_4_CV.keras',
                                                                                            name2='AE_RNN_Model_5_CV.keras')
"""

In [ ]:
CNN_LSM_Model_1_GA_CV, AE_LSM_Model_2_CV, cnn_rnn_test_ds_cv, all_histories1, all_histories2 = cross_session(CNN_LSM_Model_1_GA, 
                                                                                            AE_LSM_Model_2,
                                                                                            name1='CNN_LSM_Model_1_GA_CV.keras',
                                                                                            name2='AE_LSM_Model_2_CV.keras')

In [ ]:
def print_graphsCV(all_histories):
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    for session_id, metrics in all_histories.items():
        ax1.plot(metrics['accuracy'], marker='o', label=f'Session {session_id}')
        ax1.set_title(f'Accuracy Per Fold')
        ax1.legend()
        
        ax2.plot(metrics['val_accuracy'], marker='o', label=f'Session {session_id}')
        ax2.set_title(f'Val Accuracy Per Fold')
        ax2.legend()
    
        ax3.plot(metrics['loss'], marker='o', label=f'Session {session_id}')
        ax3.set_title(f'Loss Per Fold')
        ax3.legend()
    
        ax4.plot(metrics['val_loss'], marker='o', label=f'Session {session_id}')
        ax4.set_title(f'Val loss Per Fold')
        ax4.legend()
    
    plt.tight_layout()
    plt.show()

#print("Graphs from AE_RNN_Model_4_CV" )
#print_graphsCV(all_histories1)
#print("Graphs from AE_RNN_Model_5_CV" )
#print_graphsCV(all_histories2)

In [ ]:
print("Graphs from CNN_LSM_Model_1_GA_CV" )
print_graphsCV(all_histories1)
print("Graphs from AE_LSM_Model_2_CV" )
print_graphsCV(all_histories2)

In [ ]:
"""
CNN_RNN_Model_1.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate = 3e-4),
    loss = loss_object,
    metrics = [metric_SPCC]
)"""

In [ ]:
#Compiler(CNN_RNN_Model_5)
#Compiler(AE_RNN_Model_1)
#Compiler(AE_RNN_Model_2)

In [ ]:
"""
AE_RNN_Model_2.fit(
    cnn_rnn_train_ds, 
    epochs = 50,
    validation_data = cnn_rnn_val_ds, 
    callbacks = [lr_callback, GradientMonitor(cnn_rnn_train_ds), rt_callback, early_stop_callback], 
    verbose = 1)
    """

In [ ]:
#CNN_RNN_Model_5.save('CNN_RNN_Model_5.keras')
#AE_RNN_Model_2.save('AE_RNN_Model_2.keras')

In [ ]:
#CNN_RNN_Model_1.fit(
#    cnn_rnn_train_ds, 
#    epochs = 50,
#    validation_data = cnn_rnn_val_ds, 
#    callbacks = [lr_callback, GradientMonitor(cnn_rnn_train_ds)], 
#    verbose = 1)

In [ ]:
#CNN_RNN_Model_1.save('CNN_RNN_Model_1.keras')

In [ ]:
# #Compile and train the second LSM model
# #And the GA version

# LSM_Model_2.compile(
#    optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-3),
#    loss = loss_object,
#    metrics = [metric_SPCC]
# )
# LSM_Model_2_GA.compile(
#     optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-3),
#     loss = loss_object,
#     metrics = [metric_SPCC]
# )

# LSM_Model_2.fit(
#    lsm_train_ds,
#    epochs = 50,
#    validation_data = lsm_val_ds,
#    callbacks = [early_stop_callback, GradientMonitor(lsm_train_ds)]
# )

# LSM_Model_2_GA.fit(
#    lsm_train_ds,
#    epochs = 50,
#    validation_data = lsm_val_ds,
#    callbacks = [early_stop_callback, GradientMonitor(lsm_train_ds)]
# )

In [ ]:
# LSM_Model_3.compile(
#     optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-3),
#     loss = loss_object,
#     metrics = [metric_SPCC]
# )

# LSM_Model_3_GA.compile(
#     optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-3),
#     loss = loss_object,
#     metrics = [metric_SPCC]
# )

# LSM_Model_3.fit(
#     lsm_train_ds,
#     epochs = 50,
#     validation_data = lsm_val_ds,
#     callbacks = [early_stop_callback, GradientMonitor(lsm_train_ds)]
# )

# LSM_Model_3_GA.fit(
#     lsm_train_ds,
#     epochs = 50,
#     validation_data = lsm_val_ds,
#     callbacks = [early_stop_callback, GradientMonitor(lsm_train_ds)]
# )

In [ ]:
# LSM_Model_4.compile(
#     optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-3),
#     loss = loss_object,
#     metrics = [metric_SPCC]
# )

# LSM_Model_4_GA.compile(
#     optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-3),
#     loss = loss_object,
#     metrics = [metric_SPCC]
# )

# LSM_Model_4.fit(
#     lsm_train_ds,
#     epochs = 50,
#     validation_data = lsm_val_ds,
#     callbacks = [early_stop_callback, GradientMonitor(lsm_train_ds)]
# )

# LSM_Model_4_GA.fit(
#     lsm_train_ds,
#     epochs = 50,
#     validation_data = lsm_val_ds,
#     callbacks = [early_stop_callback, GradientMonitor(lsm_train_ds)]
# )

In [ ]:
# LSM_Model_5.compile(
#     optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-3),
#     loss = loss_object,
#     metrics = [metric_SPCC]
# )

# LSM_Model_5_GA.compile(
#     optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-3),
#     loss = loss_object,
#     metrics = [metric_SPCC]
# )

# LSM_Model_5.fit(
#     lsm_train_ds,
#     epochs = 50,
#     validation_data = lsm_val_ds,
#     callbacks = [early_stop_callback, GradientMonitor(lsm_train_ds)]
# )

# LSM_Model_5_GA.fit(
#     lsm_train_ds,
#     epochs = 50,
#     validation_data = lsm_val_ds,
#     callbacks = [early_stop_callback, GradientMonitor(lsm_train_ds)]
# )

In [ ]:
# LSM_Model_2.save('LSM_Model_2.keras')
# LSM_Model_3.save('LSM_Model_3.keras')
# LSM_Model_4.save('LSM_Model_4.keras')
# LSM_Model_5.save('LSM_Model_5.keras')

# LSM_Model_2_GA.save('LSM_Model_2_GA.keras')
# LSM_Model_3_GA.save('LSM_Model_3_GA.keras')
# LSM_Model_4_GA.save('LSM_Model_4_GA.keras')
# LSM_Model_5_GA.save('LSM_Model_5_GA.keras')

In [ ]:
# CNN_LSM_Model_1.compile(
#     optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-3),
#     loss = loss_object,
#     metrics = [metric_SPCC]
# )

# CNN_LSM_Model_1_GA.compile(
#     optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-3),
#     loss = loss_object,
#     metrics = [metric_SPCC]
# )

# CNN_LSM_Model_1.fit(
#     cnn_rnn_train_ds,
#     epochs = 50,
#     validation_data = cnn_rnn_val_ds,
#     callbacks = [early_stop_callback, GradientMonitor(cnn_rnn_train_ds)]
# )

# CNN_LSM_Model_1_GA.fit(
#     cnn_rnn_train_ds,
#     epochs = 50,
#     validation_data = cnn_rnn_val_ds,
#     callbacks = [early_stop_callback, GradientMonitor(cnn_rnn_train_ds)]
# )

In [ ]:
# CNN_LSM_Model_2.compile(
#     optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-3),
#     loss = loss_object,
#     metrics = [metric_SPCC]
# )

# CNN_LSM_Model_2_GA.compile(
#     optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-3),
#     loss = loss_object,
#     metrics = [metric_SPCC]
# )

# CNN_LSM_Model_2.fit(
#     cnn_rnn_train_ds,
#     epochs = 50,
#     validation_data = cnn_rnn_val_ds,
#     callbacks = [early_stop_callback, GradientMonitor(cnn_rnn_train_ds)]
# )

# CNN_LSM_Model_2_GA.fit(
#     cnn_rnn_train_ds,
#     epochs = 50,
#     validation_data = cnn_rnn_val_ds,
#     callbacks = [early_stop_callback, GradientMonitor(cnn_rnn_train_ds)]
# )

In [ ]:
# CNN_LSM_Model_1.save('CNN_LSM_Model_1.keras')
# CNN_LSM_Model_2.save('CNN_LSM_Model_2.keras')

# CNN_LSM_Model_1_GA.save('CNN_LSM_Model_1_GA.keras')
# CNN_LSM_Model_2_GA.save('CNN_LSM_Model_2_GA.keras')

In [ ]:
# AE_LSM_Model_1.compile(
#     optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-3),
#     loss = loss_object,
#     metrics = [metric_SPCC]
# )

# AE_LSM_Model_1.fit(
#     cnn_rnn_train_ds,
#     epochs = 50,
#     validation_data = cnn_rnn_val_ds,
#     callbacks = [early_stop_callback, GradientMonitor(cnn_rnn_train_ds)]
# )

In [ ]:
# AE_LSM_Model_2.compile(
#     optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-3),
#     loss = loss_object,
#     metrics = [metric_SPCC]
# )

# AE_LSM_Model_2.fit(
#     cnn_rnn_train_ds,
#     epochs = 50,
#     validation_data = cnn_rnn_val_ds,
#     callbacks = [early_stop_callback, GradientMonitor(cnn_rnn_train_ds)]
# )

In [ ]:
# AE_LSM_Model_3.compile(
#     optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-3),
#     loss = loss_object,
#     metrics = [metric_SPCC]
# )

# AE_LSM_Model_3.fit(
#     cnn_rnn_train_ds,
#     epochs = 50,
#     validation_data = cnn_rnn_val_ds,
#     callbacks = [early_stop_callback, GradientMonitor(cnn_rnn_train_ds)]
# # )

In [ ]:
#AE_LSM_Model_1.save('AE_LSM_Model_1.keras')
#AE_LSM_Model_2.save('AE_LSM_Model_2.keras')
#AE_LSM_Model_3.save('AE_LSM_Model_3.keras')

# Testing 

In the following section we will test our models by using 3 helper functions:
* **load_model** simply loads and compile models. It's not necessary to use this function when we are training our models inside the notebook.
* **test_model** uses the evaluate tool to provide a general accuracy of our model to control its performance
* **create_confusion_matrix** computes the confusion matrix by using the exact predictions on the model. From this matrix, we obtain the accuracies per gesture, and the global average accuracy.

DISCLAIMER: This notebook is a combined version of all our work. However, not all our models are tested in the following lines, most of them were trained directly after training. For this reason, in our Demo notebook we directly incuded our best 

In [ ]:
#Helper functions to load, test and print statistics of different models

def load_model(path,lr):
    #Importing and compiling it

    custom_map = {'LIFCell' : LIFCell}
    
    Model_test = tf.keras.models.load_model(path, compile=False, custom_objects = custom_map)
    Model_test.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate = lr),
    loss = loss_object,
    metrics = [metric_SPCC])
    return Model_test

def test_model(model, test_ds):
    #Evaluate model on testing set
    test_results = model.evaluate(test_ds, verbose=1, return_dict=True)

    return test_results 

def create_confusion_matrix(model, test_ds):
    # 1. Get raw logits predictions
    y_pred_logits = model.predict(test_ds)
    
    # 2. Convert logits and obtains labels by iterating over all the batches
    y_pred_classes = np.argmax(y_pred_logits, axis=1)
    y_true_classes = np.concatenate([y for x, y in test_ds], axis=0) 
    
    # 3. Define gesture labels
    gesture_names = ['Pinch Index', 'Pinch Pinky', 'Finger Slide', 'Finger Rub', 'Slow Swipe', 
                     'Fast Swipe', 'Push', 'Pull', 'Palm Tilt', 'Circle', 'Palm Hold']
    
    # 4. Create the matrix
    cm = confusion_matrix(y_true_classes, y_pred_classes)
    
    # 5. Plot
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=gesture_names, 
                yticklabels=gesture_names)
    plt.title(f"Confusion Matrix: {model.name}")
    plt.ylabel('True Gesture')
    plt.xlabel('Predicted Gesture')
    plt.show()

    #6. Compute accuracies per gesture
    print("Accuracies per gesture")
    accuracies= []
    for i in range(11):
        tp = cm[i,i]
        total = sum(cm[i])
        accuracy = round(tp/total, 4)
        accuracies.append(accuracy)
        

        print(f"{gesture_names[i]}: {accuracy}")
    avg_acc = sum(accuracies)/11
    print(f"Global average accuracy: {avg_acc:.4f}")

    trainable_count = np.sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
    non_trainable_count = np.sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])

    print(f"Parameter count: {trainable_count + non_trainable_count:,}")
    print(f"Trainable: {trainable_count:,}")
    print(f"Non-trainable: {non_trainable_count:,}")

    return cm, accuracies

In [ ]:
import kagglehub

#Paths to import the trained models
path = kagglehub.model_download("xxdiegoalonsoxx/model-1/tensorFlow2/cnnrnn-1/2")
print("Path to model files:", path)
path = kagglehub.model_download("xxdiegoalonsoxx/model-1/tensorFlow2/default/2")
print("Path to model files:", path)
path = kagglehub.model_download("xxdiegoalonsoxx/model-1/tensorFlow2/ae_rnn/2")
print("Path to model files:", path)
path = kagglehub.model_download("xxdiegoalonsoxx/model-1/tensorFlow2/lsm2")
print("Path to model files:", path)

## Testing CNN_LSM_Model_1_GA_CV y AE_LSM_Model_2_CV

In [ ]:
#Testing model 5
#model_path = "/kaggle/input/model-1/tensorflow2/ae_rnn/2/AE_RNN_Model_2.keras"
#learning_rate = 3e-4

test_set = cnn_rnn_test_ds_cv
model = CNN_LSM_Model_1_GA_CV #load_model(model_path, learning_rate)

test_results = test_model(model, test_set)

print(f"Accuracy from test set: {test_results['accuracy']:.4f}")
cm, acc_x_gest = create_confusion_matrix(model, test_set)

In [ ]:
#Testing model 5
#model_path = "/kaggle/input/model-1/tensorflow2/ae_rnn/2/AE_RNN_Model_2.keras"
#learning_rate = 3e-4
test_set = cnn_rnn_test_ds_cv
model = AE_LSM_Model_2_CV #load_model(model_path, learning_rate)

test_results = test_model(model, test_set)

print(f"Accuracy from test set: {test_results['accuracy']:.4f}")
cm, acc_x_gest = create_confusion_matrix(model, test_set)